<a href="https://colab.research.google.com/github/yamak493/nlf/blob/claude/kind-mendel-tdveor/mp4_to_mannequin_ja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕺 mp4 → モーション抽出 → 踊るマネキン動画（音声付き）

このノートブックは、**指定 URL からダウンロードした mp4 の「始点秒〜終点秒」の区間から人物の 3D モーションを抽出し、
そのモーションで動くマネキンの動画（元動画の音声付き mp4）を書き出す**ためのものです。

使用する技術:

| 役割 | 使うもの |
|---|---|
| 画像 → 3D 人体（頂点・関節・SMPL パラメータ） | [NLF (Neural Localizer Fields)](https://github.com/isarandi/nlf) の TorchScript モデル `v0.3.2` |
| 3D 頂点 → SMPL パラメータへのフィッティング | [SMPLFitter](https://github.com/isarandi/smplfitter)（NLF の内部でも使われています） |
| 動画の切り出し・音声の合成 | ffmpeg（`imageio-ffmpeg` に同梱のバイナリを使うので別途インストール不要） |

## 処理の流れ

1. ライブラリを自動インストール（`smplfitter` など）
2. NLF の学習済みモデル（約 470&nbsp;MB）を自動ダウンロード
3. 入力動画（mp4）を URL からダウンロード（既定: `https://made-by-free.com/night-fire.mp4`）
4. 始点秒・終点秒を指定して、その区間を切り出し（映像と音声）
5. 1 フレームずつ NLF で推論 → 人物を 1 人選んで追跡 → **モーションデータ**（SMPL の pose / betas / trans / 関節 / 頂点）を取得
6. 時間方向に平滑化して `motion.npz` に保存
7. Cascadeur 風のクリーンアップ（支点の検出 → キーフレーム化 → 支点を固定する IK）で
   **足ズレ・細かな振動**を和らげる
8. モーションを反映した**マネキンのメッシュ**を作ってレンダリング
9. 元動画の音声を合成して `mannequin_with_audio.mp4` を出力・再生・ダウンロード
10. モーションを **FBX**（`motion.fbx`：スケルトン＋スキン付きマネキン＋アニメーション）でも出力・ダウンロード

## 実行前の注意

* **GPU ランタイムが必須**です。Colab では「ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ: GPU」を選んでください（NLF は半精度で動くため CPU では実行できません）。
* 処理時間の目安（Colab T4 / 720p）: **推論 約 0.3〜0.6 秒/フレーム**。まずは **5〜10 秒程度**の区間で試してください。
* マネキンの見た目は 2 種類あります。
  * **パーツ凸包マネキン（既定）**: SMPL の公式ファイルが無くても動きます。木製デッサン人形のような見た目になります。
  * **SMPL メッシュ**: SMPL 公式配布ファイル（要ユーザー登録）がある場合のみ。人体そのままの滑らかなメッシュになります。
* ライセンス: NLF のモデルは**非商用の研究用途**で公開されています。SMPL 系ボディモデルは [smpl.is.tue.mpg.de](https://smpl.is.tue.mpg.de/) 等での登録・ライセンス同意が必要です。入力する動画は自分に権利があるものを使ってください。

---
## 1. ライブラリのインストール

必要なパッケージを自動で入れます（Colab では PyTorch は既に入っているのでそのまま使います）。
初回は 1〜2 分ほどかかります。

In [ ]:
#@title 1. セットアップ（実行するだけ） { display-mode: "form" }
import importlib.util
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
print('Colab 上で実行中:', IN_COLAB)


def pip_install(*pkgs):
    print('インストール中:', ' '.join(pkgs))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)


# --- PyTorch（Colab には最初から入っています） ---
try:
    import torch
    import torchvision  # NLF の TorchScript を読むために必須
except ImportError:
    pip_install('torch', 'torchvision')
    import torch
    import torchvision

# --- その他の依存パッケージ ---
required = [
    ('smplfitter', 'smplfitter'),   # SMPL フィッティング（NLF 内部でも使用）
    ('scipy', 'scipy'),
    ('trimesh', 'trimesh'),
    ('imageio', 'imageio'),
    ('imageio_ffmpeg', 'imageio-ffmpeg'),  # ffmpeg バイナリ同梱
    ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'),
    ('PIL', 'pillow'),
]
missing = [pkg for mod, pkg in required if importlib.util.find_spec(mod) is None]
if missing:
    pip_install(*missing)
else:
    print('依存パッケージはすべて揃っています。')

import numpy as np

# numpy 2.x では np.infty が削除されたが、pyrender など一部ライブラリがまだ使うので補う
if not hasattr(np, 'infty'):
    np.infty = np.inf

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print()
print('PyTorch:', torch.__version__, '/ NumPy:', np.__version__)
print('デバイス:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ GPU が見つかりません。NLF は half 精度で動作するため GPU が必要です。')
    print('   Colab なら「ランタイム → ランタイムのタイプを変更 → GPU」を選んでから、')
    print('   このノートブックを最初から実行し直してください。')

WORK_DIR = os.path.abspath('nlf_mannequin')
os.makedirs(WORK_DIR, exist_ok=True)
print('作業ディレクトリ:', WORK_DIR)

---
## 2. NLF 学習済みモデルのダウンロード

[NLF v0.3.2 リリース](https://github.com/isarandi/nlf/releases/tag/v0.3.2) の
`nlf_l_multi_0.3.2.torchscript`（約 470&nbsp;MB / EfficientNetV2-L バックボーン）を取得します。

このモデルは **人物検出 → 3D 頂点・関節の推定 → SMPL パラメータへのフィッティング（SMPLFitter）** までを
1 つの TorchScript にまとめたものです。一度ダウンロードすれば以降のセル実行では再利用されます。

In [ ]:
#@title 2. NLF モデルの自動ダウンロードと読み込み { display-mode: "form" }
import time
import urllib.request

MODEL_URL = 'https://github.com/isarandi/nlf/releases/download/v0.3.2/nlf_l_multi_0.3.2.torchscript'
MODEL_PATH = os.path.join(WORK_DIR, 'nlf_l_multi_0.3.2.torchscript')
MIN_EXPECTED_BYTES = 300 * 1024 ** 2


def download(url, path, min_bytes=0):
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        print(f'既にダウンロード済み: {path} ({os.path.getsize(path) / 1024 ** 2:.0f} MB)')
        return path
    tmp = path + '.part'
    print('ダウンロード中:', url)
    t0 = time.time()
    with urllib.request.urlopen(url) as resp, open(tmp, 'wb') as f:
        total = int(resp.headers.get('Content-Length', 0))
        done = 0
        while True:
            chunk = resp.read(1024 * 1024)
            if not chunk:
                break
            f.write(chunk)
            done += len(chunk)
            if total:
                bar = '█' * int(30 * done / total)
                print(f'\r  [{bar:<30}] {done / 1024 ** 2:7.0f} / {total / 1024 ** 2:.0f} MB',
                      end='', flush=True)
    print()
    os.replace(tmp, path)
    print(f'完了（{time.time() - t0:.0f} 秒）:', path)
    return path


download(MODEL_URL, MODEL_PATH, MIN_EXPECTED_BYTES)

print('モデルを読み込み中…（30 秒ほどかかります）')
nlf_model = torch.jit.load(MODEL_PATH).to(DEVICE).eval()
print('読み込み完了 ✅')

---
## 3.（任意）SMPL 公式ボディモデルの用意

* **何もしなくても動きます。** その場合は SMPL の頂点を体のパーツごとに凸包（convex hull）で包んだ
  「デッサン人形風マネキン」でレンダリングします（メッシュの面情報が不要な方式です）。
* SMPL の公式ファイルがあると、**人体メッシュそのもの**をマネキンとして描画でき、さらに
  SMPLFitter で「全フレーム共通の体型（betas）」に整えるリフィットも実行できます。

公式ファイルは [smpl.is.tue.mpg.de](https://smpl.is.tue.mpg.de/) でのユーザー登録とライセンス同意が必要です。
登録済みなら、下の `DOWNLOAD_SMPL_MODEL` を `True` にして実行すると、`smplfitter` のダウンローダ経由で
メールアドレスとパスワードを聞かれ、自動で配置されます（入力内容はどこにも保存されません）。
既に手元にファイルがある場合は `body_models/smpl/` 以下に置いてください。

In [ ]:
#@title 3. ボディモデルの検出（任意ダウンロード） { display-mode: "form" }
DOWNLOAD_SMPL_MODEL = False  #@param {type:"boolean"}

BODY_MODELS_DIR = os.path.join(WORK_DIR, 'body_models')
os.makedirs(BODY_MODELS_DIR, exist_ok=True)
os.environ['SMPLFITTER_BODY_MODELS'] = BODY_MODELS_DIR
# 想定する配置: body_models/smpl/basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl

if DOWNLOAD_SMPL_MODEL:
    import getpass
    from pathlib import Path
    from urllib.parse import quote
    try:
        from smplfitter.download import _download_smpl, _make_opener
        email = input('MPI (smpl.is.tue.mpg.de) の登録メールアドレス: ')
        password = getpass.getpass('パスワード: ')
        auth = f'username={quote(email, safe="")}&password={quote(password, safe="")}'.encode()
        _download_smpl(_make_opener(), auth, Path(BODY_MODELS_DIR))
    except Exception as e:
        print('⚠️ ダウンロードに失敗しました:', repr(e))
        print('   登録が済んでいるか、メール/パスワードが正しいかを確認してください。')
        print('   （このまま進めてもパーツ凸包マネキンで動画は作れます）')


def load_body_model(model_name='smpl', gender='neutral'):
    # 公式ボディモデルが見つかればロードする。無ければ None を返す。
    try:
        from smplfitter.pt import BodyModel
        bm = BodyModel(model_name, gender, num_betas=10)
        return bm
    except Exception as e:
        print(f'SMPL 公式ファイルは見つかりませんでした（{type(e).__name__}）。')
        return None


BODY_MODEL = load_body_model('smpl')
SMPL_FACES = None if BODY_MODEL is None else np.asarray(BODY_MODEL.faces, np.int32)
if SMPL_FACES is None:
    print('→ マネキンは「パーツ凸包」方式で作ります（公式ファイル不要）。')
else:
    print(f'→ SMPL メッシュが使えます（面数 {len(SMPL_FACES)}）。')

---
## 4. 入力動画（mp4）のダウンロード

* `VIDEO_URL`（既定: `https://made-by-free.com/night-fire.mp4`）から mp4 をダウンロードして入力にします。
* ダウンロード済みのファイルがあれば再ダウンロードはしません。
* 手元のファイルを使いたい場合は `VIDEO_PATH` にパスを書いてください（`VIDEO_URL` より優先されます）。


In [ ]:
#@title 4. 動画をダウンロード { display-mode: "form" }
import shutil
import urllib.parse
import urllib.request

VIDEO_URL = 'https://made-by-free.com/night-fire.mp4'  #@param {type:"string"}
VIDEO_PATH = ''  #@param {type:"string"}

if VIDEO_PATH:
    assert os.path.exists(VIDEO_PATH), f'ファイルが見つかりません: {VIDEO_PATH}'
else:
    assert VIDEO_URL, 'VIDEO_URL か VIDEO_PATH を指定してください。'
    fname = os.path.basename(urllib.parse.urlparse(VIDEO_URL).path) or 'input.mp4'
    VIDEO_PATH = os.path.join(WORK_DIR, fname)
    if not os.path.exists(VIDEO_PATH) or os.path.getsize(VIDEO_PATH) == 0:
        print('ダウンロード中:', VIDEO_URL)
        tmp_path = VIDEO_PATH + '.part'
        req = urllib.request.Request(VIDEO_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resp, open(tmp_path, 'wb') as f:
            shutil.copyfileobj(resp, f)
        os.replace(tmp_path, VIDEO_PATH)

print('入力動画:', VIDEO_PATH, f'({os.path.getsize(VIDEO_PATH) / 1024 ** 2:.1f} MB)')

---
## 5. 動画の情報を確認して、切り出す区間などを設定

ここで **始点秒数 `START_SEC`・終点秒数 `END_SEC`** を指定します。まずは 5〜10 秒程度で試すのがおすすめです。

### 切り出し・人物

| 設定 | 意味 |
|---|---|
| `START_SEC` / `END_SEC` | 切り出す区間（秒）。`END_SEC = 0` なら動画の最後まで |
| `TARGET_FPS` | 処理する fps（`0` で元動画のまま）。小さくすると速くなります |
| `MAX_HEIGHT` | 推論時に縮小する高さの上限（720 程度が速度と精度のバランス◎） |
| `PERSON_SELECT` | 最初のフレームでどの人物を主役にするか（`largest`=一番大きく写っている人 / `center`=画面中央の人） |

### 推論（GPU の使い方と精度）

| 設定 | 意味 |
|---|---|
| `BATCH_SIZE` | 一度に GPU へ送るフレーム数 |
| `NUM_AUG` | 1 人あたりのテスト時データ拡張（TTA）の枚数。**奇数のみ**（偶数だと拡張の振り方が左右非対称になります） |
| `ANTIALIAS` | クロップを作るときの超サンプリング倍率。小さく写っている人物に効きます（NLF 本体のコメントに「4 は精度が上がることがある」とあります） |
| `DETECTOR_THRESHOLD` | 人物検出のしきい値。検出が途切れるときは 0.15 程度まで下げてください |

NLF が GPU に流すのは「**フレーム数 × 人数 × `NUM_AUG`** 枚のクロップ（384×384）」です。
1 人しか写っていない動画で `BATCH_SIZE=4`・`NUM_AUG=1` だと 4 枚しか流れず、GPU がほとんど遊びます。
**GPU メモリが余っているときは、まず `BATCH_SIZE` を増やし、次に `NUM_AUG` を増やしてください。**

`NUM_AUG` は、明るさ・面内回転・拡大率・左右反転を変えた複数のクロップで推論し、
**不確実性で重み付けした幾何中央値**で統合する仕組みです（NLF 本体の機能）。
つまり「体の細かい震え」の原因であるフレームごとの推定ノイズを、後処理ではなく**発生源で**減らせます。
所要時間はおおよそ `NUM_AUG` に比例します（1 → 3 で約 2〜3 倍）。メモリが足りなければ自動でバッチを分割して再試行します。

### モーションの整形（震え・足滑り対策）

| 設定 | 意味 |
|---|---|
| `MEDIAN_WINDOW` | 単発の外れフレームを除く中央値フィルタの窓（フレーム。1 で無効） |
| `SMOOTH_CUTOFF_HZ` | 手足の回転と全身の位置のローパス遮断周波数。小さいほど滑らか、大きいほどキレが残る |
| `ROOT_CUTOFF_HZ` | **全身の向き**のローパス遮断周波数。胴全体が細かく震えるときはここを下げます |
| `DEPTH_SCALE_FIX` | **体の大きさのブレ対策**（既定オン）。体型を 1 本に固定したぶん、各フレームの位置をカメラ原点から伸縮して、元動画での見かけの大きさを保ちます |
| `DEPTH_EXTRA_SMOOTH` | 奥行き（z）だけ強めに平滑化。奥行きのブレは減りますが、**前後に動く振付では足が滑りやすくなる**ので既定はオフ（奥行きのドリフトは 8c の接地補正で直します） |
| `FOOT_LOCK` | 接地している足が地面上で止まるよう全身の位置を補正（足滑り対策） |
| `ROOT_MOTION` | `locked`=その場で踊る（低周波のドリフトを除去）/ `smoothed`=移動を残して滑らかに / `full`=補正後の移動をそのまま |

平滑化はゼロ位相フィルタ（前後両方向にかける）なので**動きが遅れません**。

### 出力

| 設定 | 意味 |
|---|---|
| `MANNEQUIN_STYLE` | `auto`（公式 SMPL があればメッシュ、無ければパーツ凸包）/ `parts` / `smpl_mesh` |
| `CAMERA_MODE` | `fit`=元の視点のまま人物が画面いっぱいに映るよう自動フレーミング / `original`=元動画と同じ画角 |
| `VIEW_AZIMUTH_DEG` | マネキンを縦軸まわりに回して別角度から見る（度） |
| `SHOW_FULCRUM_MARKERS` | 8d が検出した**支点**（接地している足・手）をプレビュー動画にマーカーで重ねる（セル 11 の設定） |
| `SIDE_BY_SIDE` | 出力の左に元動画、右にマネキンを並べる（幅が 2 倍になります）。位置をそのまま見比べたいときは `CAMERA_MODE='original'` と併用 |

In [ ]:
#@title 5. 区間・出力の設定 { display-mode: "form" }
# --- 切り出す区間と入力の解像度 ---
START_SEC = 30.0  #@param {type:"number"}
END_SEC = 60.0  #@param {type:"number"}
TARGET_FPS = 30  #@param {type:"integer"}
MAX_HEIGHT = 720  #@param {type:"integer"}
PERSON_SELECT = "largest"  #@param ["largest", "center"]

# --- 推論（GPU が余っているなら BATCH_SIZE → NUM_AUG の順に増やす） ---
BATCH_SIZE = 128  #@param {type:"integer"}
NUM_AUG = 3  #@param [1, 3, 5, 7] {type:"raw"}
ANTIALIAS = 2  #@param [1, 2, 4] {type:"raw"}
DETECTOR_THRESHOLD = 0.25  #@param {type:"number"}

# --- モーションの整形（震え・足滑り対策） ---
MEDIAN_WINDOW = 3  #@param {type:"integer"}
SMOOTH_CUTOFF_HZ = 6.0  #@param {type:"number"}
ROOT_CUTOFF_HZ = 3.0  #@param {type:"number"}
DEPTH_SCALE_FIX = True  #@param {type:"boolean"}
DEPTH_EXTRA_SMOOTH = False  #@param {type:"boolean"}
FOOT_LOCK = True  #@param {type:"boolean"}
ROOT_MOTION = "locked"  #@param ["locked", "smoothed", "full"]

# --- 出力 ---
MANNEQUIN_STYLE = "auto"  #@param ["auto", "parts", "smpl_mesh"]
CAMERA_MODE = "fit"  #@param ["fit", "original"]
VIEW_AZIMUTH_DEG = 0  #@param {type:"slider", min:-180, max:180, step:15}
SHOW_FLOOR = True  #@param {type:"boolean"}
SIDE_BY_SIDE = False  #@param {type:"boolean"}
OUT_HEIGHT = 720  #@param {type:"integer"}

# ---- ffmpeg まわりのユーティリティ（imageio-ffmpeg 同梱のバイナリを使う） ----
import imageio.v2 as imageio
import imageio_ffmpeg


def ffmpeg_exe():
    try:
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return 'ffmpeg'


def run_ffmpeg(args, check=True):
    p = subprocess.run([ffmpeg_exe(), '-hide_banner', *args],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if check and p.returncode != 0:
        raise RuntimeError('ffmpeg 失敗:\n' + p.stdout[-3000:])
    return p


def probe(path):
    with imageio.get_reader(path) as r:
        meta = r.get_meta_data()
    info = run_ffmpeg(['-i', path], check=False).stdout
    return dict(fps=float(meta.get('fps') or 30.0),
                duration=float(meta.get('duration') or 0.0),
                size=tuple(meta.get('size') or (0, 0)),
                has_audio=('Audio:' in info))


SRC_INFO = probe(VIDEO_PATH)
print('入力動画:', SRC_INFO)

duration = SRC_INFO['duration']
START_SEC = max(0.0, float(START_SEC))
END_SEC = float(END_SEC)
if END_SEC <= 0 or (duration and END_SEC > duration):
    END_SEC = duration if duration else END_SEC
assert END_SEC > START_SEC, '終点秒数は始点秒数より後にしてください。'
FPS = float(TARGET_FPS) if TARGET_FPS and TARGET_FPS > 0 else SRC_INFO['fps']
n_expected = int(round((END_SEC - START_SEC) * FPS))
print(f'切り出す区間: {START_SEC:.2f} 秒 〜 {END_SEC:.2f} 秒'
      f'（{END_SEC - START_SEC:.2f} 秒 / 約 {n_expected} フレーム @ {FPS:g} fps）')
if n_expected > 900:
    print('⚠️ フレーム数が多いので時間がかかります。まずは短い区間で試すことをおすすめします。')

---
## 6. 指定区間の切り出し（映像 + 音声）

映像は推論しやすいように `MAX_HEIGHT` 以下に縮小し、`TARGET_FPS` に揃えます。
音声は同じ区間を `m4a` として取り出し、最後にマネキン動画へ合成します（音声トラックが無い動画でもそのまま進みます）。

In [ ]:
#@title 6. 区間を切り出す { display-mode: "form" }
SEGMENT_MP4 = os.path.join(WORK_DIR, 'segment.mp4')
SEGMENT_AUDIO = os.path.join(WORK_DIR, 'segment.m4a')

vf = [f'fps={FPS}']
if MAX_HEIGHT and MAX_HEIGHT > 0:
    # 高さが MAX_HEIGHT を超えるときだけ縮小（幅は 2 の倍数に丸める）
    vf.append(f"scale='trunc(iw*min(1,{MAX_HEIGHT}/ih)/2)*2':'trunc(ih*min(1,{MAX_HEIGHT}/ih)/2)*2'")

run_ffmpeg(['-y', '-loglevel', 'error', '-ss', f'{START_SEC:.3f}', '-to', f'{END_SEC:.3f}',
            '-i', VIDEO_PATH, '-an', '-vf', ','.join(vf),
            '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '18', '-pix_fmt', 'yuv420p',
            SEGMENT_MP4])
SEG_INFO = probe(SEGMENT_MP4)
print('切り出した映像:', SEG_INFO)

AUDIO_PATH = None
if SRC_INFO['has_audio']:
    try:
        run_ffmpeg(['-y', '-loglevel', 'error', '-ss', f'{START_SEC:.3f}', '-to', f'{END_SEC:.3f}',
                    '-i', VIDEO_PATH, '-vn', '-c:a', 'aac', '-b:a', '192k', SEGMENT_AUDIO])
        if os.path.getsize(SEGMENT_AUDIO) > 0:
            AUDIO_PATH = SEGMENT_AUDIO
    except Exception as e:
        print('音声の取り出しに失敗しました:', repr(e))
print('音声:', AUDIO_PATH or 'なし（無音の動画を出力します）')

---
## 7. モーション抽出（NLF 推論）

各フレームを NLF に通して、人物ごとに

* `pose` … SMPL の関節回転（回転ベクトル 24×3）
* `betas` … 体型パラメータ（10 次元）
* `trans` … 全身の位置（m）
* `joints3d` / `vertices3d` … カメラ座標系の 3D 関節・頂点（mm 単位、x=右 / y=下 / z=奥）

を得ます。NLF は内部で **SMPLFitter** を使い、推定した非パラメトリックな頂点・関節に SMPL を当てはめています。

複数人が写っている場合は、**最初のフレームで選んだ人物に最も近い検出を毎フレーム追跡**して 1 人分だけを取り出します。
検出できなかったフレームは後で前後から補間します。

In [ ]:
#@title 7. NLF でモーションを抽出 { display-mode: "form" }
from tqdm.auto import tqdm

BODY_MODEL_NAME = 'smpl'  # smplx も指定できますが、このノートブックは smpl 前提です
N_VERTS, N_JOINTS = 6890, 24
N_PREVIEW = 4  # あとで重ね描画チェックに使うフレーム数

# クロップ単位のチャンクサイズ。num_aug より小さいとチャンク分割が無効になり
# 一気に全クロップを流してしまう（メモリ不足の原因）ので下限を設ける。
INTERNAL_BATCH_SIZE = max(64, int(NUM_AUG) * 4)
print(f'1 回の呼び出しで GPU に流すクロップ: 最大 {int(BATCH_SIZE) * int(NUM_AUG)} 枚 '
      f'(BATCH_SIZE={BATCH_SIZE} x NUM_AUG={NUM_AUG}, チャンク上限 {INTERNAL_BATCH_SIZE})')


def pick_first_person(boxes, image_w):
    # boxes: (n, 5) = x, y, w, h, score
    areas = boxes[:, 2] * boxes[:, 3]
    if PERSON_SELECT == 'center':
        # ある程度大きく写っている人の中で、画面中央に最も近い人を選ぶ
        big = np.flatnonzero(areas >= 0.3 * areas.max())
        cx = boxes[big, 0] + boxes[big, 2] / 2
        return int(big[np.argmin(np.abs(cx - image_w / 2))])
    return int(np.argmax(areas))


frames_pose, frames_betas, frames_trans = [], [], []
frames_joints, frames_verts, frames_box, frames_uncert = [], [], [], []
valid = []
preview = {}

reader = imageio.get_reader(SEGMENT_MP4)
prev_trans = None
batch_imgs, batch_idx = [], []
frame_count = 0
preview_at = set(np.linspace(0, max(n_expected - 1, 0), N_PREVIEW).astype(int).tolist())


def flush(batch_imgs, batch_idx):
    global prev_trans
    if not batch_imgs:
        return
    try:
        images = torch.from_numpy(np.stack(batch_imgs)).permute(0, 3, 1, 2).contiguous().to(DEVICE)
        with torch.inference_mode(), torch.device(DEVICE):
            pred = nlf_model.detect_smpl_batched(
                images, model_name=BODY_MODEL_NAME,
                detector_threshold=float(DETECTOR_THRESHOLD),
                internal_batch_size=INTERNAL_BATCH_SIZE, num_aug=int(NUM_AUG),
                antialias_factor=int(ANTIALIAS))
    except RuntimeError as e:
        # メモリ不足のときはバッチを半分に割ってやり直す
        if 'out of memory' not in str(e).lower() or len(batch_imgs) == 1:
            raise
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        half = len(batch_imgs) // 2
        flush(batch_imgs[:half], batch_idx[:half])
        flush(batch_imgs[half:], batch_idx[half:])
        return
    for k in range(len(batch_idx)):
        boxes = pred['boxes'][k].detach().float().cpu().numpy()
        n_people = len(boxes)
        if n_people == 0:
            frames_pose.append(np.zeros(72, np.float32))
            frames_betas.append(np.zeros(10, np.float32))
            frames_trans.append(np.zeros(3, np.float32))
            frames_joints.append(np.zeros((N_JOINTS, 3), np.float32))
            frames_verts.append(np.zeros((N_VERTS, 3), np.float32))
            frames_box.append(np.zeros(5, np.float32))
            frames_uncert.append(np.nan)
            valid.append(False)
            continue
        trans = pred['trans'][k].detach().float().cpu().numpy()
        if prev_trans is None:
            j = pick_first_person(boxes, images.shape[3])
        else:
            d = np.linalg.norm(trans - prev_trans[None], axis=-1)
            j = int(np.argmin(d))
            if d[j] > 1.5:  # 追跡が外れたと判断したら一番大きい人に戻す
                j = int(np.argmax(boxes[:, 2] * boxes[:, 3]))
        prev_trans = trans[j]
        frames_pose.append(pred['pose'][k][j].detach().float().reshape(-1).cpu().numpy())
        frames_betas.append(pred['betas'][k][j].detach().float().cpu().numpy())
        frames_trans.append(trans[j])
        frames_joints.append(pred['joints3d'][k][j].detach().float().cpu().numpy())
        frames_verts.append(pred['vertices3d'][k][j].detach().float().cpu().numpy())
        frames_box.append(boxes[j])
        frames_uncert.append(
            float(pred['joint_uncertainties'][k][j].detach().float().mean().cpu()))
        valid.append(True)


pbar = tqdm(total=n_expected, desc='推論中')
for frame in reader:
    frame = np.asarray(frame)[..., :3]
    if frame_count in preview_at:
        preview[frame_count] = frame.copy()
    batch_imgs.append(frame)
    batch_idx.append(frame_count)
    frame_count += 1
    if len(batch_imgs) >= max(1, int(BATCH_SIZE)):
        flush(batch_imgs, batch_idx)
        pbar.update(len(batch_idx))
        batch_imgs, batch_idx = [], []
flush(batch_imgs, batch_idx)
pbar.update(len(batch_idx))
pbar.close()
reader.close()

valid = np.array(valid, bool)
motion_raw = dict(
    pose=np.stack(frames_pose).reshape(len(valid), -1, 3),
    betas=np.stack(frames_betas),
    trans=np.stack(frames_trans),
    joints3d=np.stack(frames_joints),
    vertices3d=np.stack(frames_verts),
    boxes=np.stack(frames_box),
)
N_FRAMES = len(valid)
UNCERTAINTY = np.array(frames_uncert, np.float32)
print(f'{N_FRAMES} フレーム処理 / 人物を検出できたフレーム: {int(valid.sum())}')
assert valid.any(), '人物が 1 人も検出できませんでした。区間や検出しきい値を変えてみてください。'
print('pose:', motion_raw['pose'].shape, ' vertices3d:', motion_raw['vertices3d'].shape, '(mm)')
if valid.any():
    u = UNCERTAINTY[valid]
    print(f'関節の推定不確実性: 中央値 {np.median(u):.0f} mm / 最大 {u.max():.0f} mm '
          f'(大きいほど推定が不安定なフレーム)')

---
## 8. モーションデータの整形と保存

NLF は 1 フレームずつ独立に推定するため、そのままだと**体が細かく震えます**。
ここでは頂点ではなく **SMPL のパラメータ（関節の回転・全身の位置・体型）を整えてから、
体モデルで頂点を作り直します**。こうすると骨の長さが厳密に一定になり、手足の伸び縮みによる震えが消えます。

1. 人物が検出できなかったフレームを前後から線形補間
2. **体型 `betas` をシーケンス全体の中央値 1 本に固定**（フレームごとの体型のゆらぎを除去）
3. **体型を固定したぶんの奥行き補正**（`DEPTH_SCALE_FIX`）
   * 単眼推定では「体の大きさ」と「奥行き」がセットで決まります。大きめの体型と推定したフレームでは
     人物を遠くに置いて画像上の大きさを合わせているので、体型だけを中央値に差し替えると、
     そのフレームでは**マネキンが小さく**（逆のフレームでは大きく）映ってしまいます
   * そこで各フレームの位置をカメラ原点から `共通体型の大きさ / そのフレームの体型の大きさ` 倍して、
     元動画での見かけの位置・大きさを保ちます（投影は厳密に変わりません）
4. **中央値フィルタ**（`MEDIAN_WINDOW`）で単発の外れフレームを除去
5. **ゼロ位相ローパス**（`scipy.signal.filtfilt`）で平滑化。前後両方向にかけるので**動きが遅れません**
   * 手足の回転と全身の位置 … `SMOOTH_CUTOFF_HZ`（既定 6 Hz）
   * **全身の向き** … `ROOT_CUTOFF_HZ`（既定 3 Hz）。胴の細かい震えはここが一番効きます
   * 奥行き `z` … `DEPTH_EXTRA_SMOOTH` が有効なら少しだけ強めに（3 Hz）
   * 位置を強く平滑化しすぎると、踏み替え（毎秒 2〜3 回）の動きまで削れて**かえって足が滑る**ため、
     位置のドリフトはフィルタではなく 8c の接地補正で直します
   * 回転は回転行列に直してから平滑化し、SVD で直交行列に戻しています
6. 整えたパラメータから**頂点と関節を再計算**（再ポーズ）
   * SMPL 公式ファイルは**不要**です。NLF の TorchScript の中に SMPL の本体（`body_models`）が
     入っているので、それをそのまま呼び出します
   * 念のため、**平滑化前**のパラメータで再ポーズしてセル 7 の結果と一致するか自己検証します。
     一致しなければ自動的に「頂点を直接平滑化する」従来方式にフォールバックします
7. `motion.npz` に保存

`motion.npz` が「抽出できたモーションデータ」です。他のツールで使いたいときはこのファイルを読み込んでください。

In [ ]:
#@title 8. 外れ値除去・平滑化・再ポーズして motion.npz に保存 { display-mode: "form" }
from scipy.ndimage import gaussian_filter1d, median_filter
from scipy.signal import butter, sosfiltfilt
from scipy.spatial.transform import Rotation

SMPL_PARENTS = np.array(
    [-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19, 20, 21], np.int32)
SMPL_JOINT_NAMES = [
    'pelvis', 'left_hip', 'right_hip', 'spine1', 'left_knee', 'right_knee', 'spine2',
    'left_ankle', 'right_ankle', 'spine3', 'left_foot', 'right_foot', 'neck', 'left_collar',
    'right_collar', 'head', 'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hand', 'right_hand']
FOOT_JOINTS = {'left': [7, 10], 'right': [8, 11]}   # 足首とつま先


def fill_gaps(arr, valid):
    # 検出できなかったフレームを、前後の有効フレームから線形補間する
    arr = np.asarray(arr, np.float32)
    idx = np.arange(len(arr))
    vi = idx[valid]
    flat = arr.reshape(len(arr), -1)[valid]
    w = np.interp(idx, vi, np.arange(len(vi)))
    lo = np.floor(w).astype(int)
    hi = np.ceil(w).astype(int)
    t = (w - lo)[:, None].astype(np.float32)
    out = flat[lo] * (1 - t) + flat[hi] * t
    return out.reshape(arr.shape)


def lowpass(x, cutoff_hz, fps, order=2):
    # ゼロ位相 Butterworth ローパス（filtfilt は前後両方向にかけるので位相遅れが出ない）
    x = np.asarray(x, np.float32)
    if not cutoff_hz or cutoff_hz <= 0 or cutoff_hz >= fps / 2:
        return x
    sos = butter(order, float(cutoff_hz) / (fps / 2), 'low', output='sos')
    if len(x) <= 3 * (2 * len(sos) + 1):
        # フレーム数が少なすぎて filtfilt が使えないので、等価なガウシアンで代用
        return gaussian_filter1d(x, fps / (2 * np.pi * cutoff_hz), axis=0,
                                 mode='nearest').astype(np.float32)
    return np.ascontiguousarray(sosfiltfilt(sos, x, axis=0), dtype=np.float32)


def median_time(x, window):
    # 時間軸だけに中央値フィルタをかけて、単発の外れフレームを落とす
    x = np.asarray(x, np.float32)
    w = int(window) | 1   # 偶数なら奇数に
    if w < 3:
        return x
    size = [1] * x.ndim
    size[0] = w
    return median_filter(x, size=size, mode='nearest')


def orthonormalize(mats):
    shape = mats.shape
    u, _, vt = np.linalg.svd(mats.reshape(-1, 3, 3))
    det = np.linalg.det(u @ vt)
    u[det < 0, :, -1] *= -1
    return (u @ vt).reshape(shape).astype(np.float32)


def clean_rotations(rotvecs, valid, fps, window, cutoff_body, cutoff_root):
    # 回転ベクトル (T, J, 3) は回転行列に直してから扱う。
    # 重要: 回転ベクトルのまま補間・平滑化してはいけない。カメラ座標系では全身の向きが
    # ほぼ 180 度回転（|回転ベクトル| ≒ π）で、表現の切れ目にちょうど乗っているため、
    # 符号が反転したフレームをまたいで線形補間すると体が 1 回転してしまう。
    rv = np.asarray(rotvecs, np.float32)
    T, J = rv.shape[:2]
    mats = Rotation.from_rotvec(rv.reshape(-1, 3)).as_matrix().reshape(T, J, 3, 3)
    mats = orthonormalize(fill_gaps(mats, valid))   # 補間も回転行列の空間で行う
    mats = orthonormalize(median_time(mats, window))
    mats = np.concatenate(
        [lowpass(mats[:, :1], cutoff_root, fps),    # 関節 0 = 全身の向き
         lowpass(mats[:, 1:], cutoff_body, fps)], axis=1)
    mats = orthonormalize(mats)
    return Rotation.from_matrix(mats.reshape(-1, 3, 3)).as_rotvec().reshape(T, J, 3).astype(
        np.float32)


def jitter_metric(joints):
    # 2 階差分の大きさ = 細かい震えの指標 [mm]
    if len(joints) < 3:
        return float('nan')
    d2 = joints[2:] - 2 * joints[1:-1] + joints[:-2]
    return float(np.linalg.norm(d2, axis=-1).mean())


def get_repose_fn():
    # 整えたパラメータから頂点・関節を作り直す関数を用意する
    try:
        bm = getattr(nlf_model.body_models, BODY_MODEL_NAME)

        def fn(pose, betas, trans):
            with torch.inference_mode():
                out = bm(pose_rotvecs=pose, shape_betas=betas, trans=trans)
            return out['vertices'], out['joints']

        return fn, 'NLF の TorchScript 内の SMPL（公式ファイル不要）'
    except Exception as e:
        print('TorchScript 内の体モデルを取り出せませんでした:', repr(e))

    if BODY_MODEL is not None:
        bm2 = BODY_MODEL.to(DEVICE)

        def fn(pose, betas, trans):
            with torch.inference_mode():
                out = bm2(pose_rotvecs=pose, shape_betas=betas, trans=trans)
            return out['vertices'], out['joints']

        return fn, 'smplfitter の公式 SMPL'
    return None, None


def repose(pose, betas, trans, fn, chunk=256):
    # 戻り値は mm（体モデルは m を返すので 1000 倍する）
    verts, joints = [], []
    for st in range(0, len(pose), chunk):
        sl = slice(st, st + chunk)
        p = torch.from_numpy(np.ascontiguousarray(pose[sl])).float().to(DEVICE)
        b = torch.from_numpy(np.ascontiguousarray(betas[sl])).float().to(DEVICE)
        t = torch.from_numpy(np.ascontiguousarray(trans[sl])).float().to(DEVICE)
        v, j = fn(p.reshape(len(p), -1), b, t)
        verts.append(v.float().cpu().numpy() * 1000.0)
        joints.append(j.float().cpu().numpy() * 1000.0)
    return np.concatenate(verts), np.concatenate(joints)


# ---- 1. 欠損補間（pose だけは clean_rotations の中で回転行列として補間する） ----
motion = {k: (v.copy() if k == 'pose' else fill_gaps(v, valid))
          for k, v in motion_raw.items()}

# ---- 2. 体型を 1 本に固定 ----
betas_const = np.median(motion_raw['betas'][valid], axis=0).astype(np.float32)
motion['betas'] = np.tile(betas_const, (N_FRAMES, 1))

# ---- 3. 再ポーズ関数の準備と自己検証 ----
repose_fn, repose_name = get_repose_fn()
if repose_fn is not None:
    check = np.flatnonzero(valid)[:8]
    try:
        v_chk, _ = repose(motion_raw['pose'][check], motion_raw['betas'][check],
                          motion_raw['trans'][check], repose_fn, chunk=8)
        err = float(np.abs(v_chk - motion_raw['vertices3d'][check]).max())
        print(f'再ポーズの自己検証: セル 7 の頂点との最大差 {err:.3f} mm（{repose_name}）')
        if not np.isfinite(err) or err > 1.0:
            print('⚠️ 一致しなかったため、再ポーズは使わず頂点を直接平滑化します。')
            repose_fn = None
    except Exception as e:
        print('⚠️ 再ポーズを実行できませんでした:', repr(e))
        repose_fn = None

# ---- 4. 体型を固定したぶんの奥行き補正 ----
# NLF はフレームごとに「体の大きさ」と「奥行き」をセットで推定している（大きい体型なら遠くに置く）。
# 体型だけを中央値に差し替えると、奥行きはそのままなので体が大きく/小さく映ってブレる。
# 共通体型での大きさとの比 r で位置をカメラ原点から伸縮すれば、画像上の見え方は元のまま保てる。
SCALE_RATIO = np.ones(N_FRAMES, np.float32)
if DEPTH_SCALE_FIX and repose_fn is not None:
    vi = np.flatnonzero(valid)
    p_raw = motion_raw['pose'][vi]
    t_raw = motion_raw['trans'][vi]
    _, j_raw = repose(p_raw, motion_raw['betas'][vi], t_raw, repose_fn)
    _, j_const = repose(p_raw, np.tile(betas_const, (len(vi), 1)), t_raw, repose_fn)

    def body_size(j):
        return np.linalg.norm(j - j.mean(1, keepdims=True), axis=-1).mean(-1)

    r = body_size(j_const) / np.maximum(body_size(j_raw), 1e-6)
    c_raw, c_const = j_raw.mean(1), j_const.mean(1)
    trans_fixed = motion_raw['trans'].copy()
    trans_fixed[vi] = t_raw + (c_raw * r[:, None] - c_const) / 1000.0
    motion['trans'] = fill_gaps(trans_fixed, valid)
    SCALE_RATIO[vi] = r

    def depth_jitter(z):
        # 1 Hz より速い奥行きの揺れ [mm]（大きさのブレに直結する）
        return float(np.std(z - lowpass(z, 1.0, FPS)))

    z_before = fill_gaps(motion_raw['trans'], valid)[:, 2] * 1000.0
    z_after = motion['trans'][:, 2] * 1000.0
    print(f'奥行き補正: 大きさの比 {r.min():.3f}〜{r.max():.3f} / '
          f'奥行きの細かい揺れ {depth_jitter(z_before):.0f} mm -> {depth_jitter(z_after):.0f} mm')
elif DEPTH_SCALE_FIX:
    print('⚠️ 再ポーズが使えないため奥行き補正はスキップしました（頂点を直接平滑化するので体型は固定されません）。')

# ---- 5. 中央値フィルタ＋ゼロ位相ローパス ----
motion['pose'] = clean_rotations(
    motion['pose'], valid, FPS, MEDIAN_WINDOW, SMOOTH_CUTOFF_HZ, ROOT_CUTOFF_HZ)
# 全身の「位置」は関節と同じ遮断周波数で。ここを下げすぎると歩幅に相当する
# 本物の動き（毎秒 2〜3 回の踏み替え）まで削れてしまい、かえって足が滑る。
# 位置のドリフトは 8c の接地補正で直す。
trans = lowpass(median_time(motion['trans'], MEDIAN_WINDOW), SMOOTH_CUTOFF_HZ, FPS)
if DEPTH_EXTRA_SMOOTH:
    # 奥行きだけは単眼推定で特に不安定なので、少しだけ強めに平滑化する
    trans[:, 2] = lowpass(trans[:, 2:3], min(3.0, SMOOTH_CUTOFF_HZ), FPS)[:, 0]
motion['trans'] = trans

# ---- 6. 再ポーズ（できなければ頂点を直接平滑化） ----
if repose_fn is not None:
    motion['vertices3d'], motion['joints3d'] = repose(
        motion['pose'], motion['betas'], motion['trans'], repose_fn)
else:
    # フォールバック: 頂点・関節に直接フィルタをかける。
    # 全身の位置だけは骨盤の軌跡から差分を作って、より低いカットオフを適用する。
    verts = lowpass(median_time(motion['vertices3d'], MEDIAN_WINDOW), SMOOTH_CUTOFF_HZ, FPS)
    joints = lowpass(median_time(motion['joints3d'], MEDIAN_WINDOW), SMOOTH_CUTOFF_HZ, FPS)
    root = joints[:, 0]
    root_target = root.copy()
    if DEPTH_EXTRA_SMOOTH:
        root_target[:, 2] = lowpass(root[:, 2:3], min(3.0, SMOOTH_CUTOFF_HZ), FPS)[:, 0]
    delta = (root_target - root)[:, None]
    motion['vertices3d'] = verts + delta
    motion['joints3d'] = joints + delta

print(f'震えの指標（関節の 2 階差分）: 整形前 {jitter_metric(motion_raw["joints3d"]):.2f} mm '
      f'-> 整形後 {jitter_metric(motion["joints3d"]):.2f} mm')

# ---- 7. 保存 ----
MOTION_NPZ = os.path.join(WORK_DIR, 'motion.npz')


def save_motion():
    # motion を書き換えるセル（8b, 8c）の最後でも呼ぶこと。
    # そうしないと .npz と描画結果が食い違う。
    np.savez_compressed(
        MOTION_NPZ,
        pose=motion['pose'].astype(np.float32),              # (T, 24, 3) 回転ベクトル [rad]
        betas=motion['betas'].astype(np.float32),            # (T, 10)
        betas_const=betas_const,                             # (10,) シーケンス共通の体型
        trans=motion['trans'].astype(np.float32),            # (T, 3) [m]
        joints3d=motion['joints3d'].astype(np.float32),      # (T, 24, 3) [mm] カメラ座標系
        vertices3d=motion['vertices3d'].astype(np.float32),  # (T, 6890, 3) [mm]
        ground_offset_mm=motion.get(
            'ground_offset_mm', np.zeros((N_FRAMES, 3), np.float32)),
        valid=valid,
        joint_uncertainty=UNCERTAINTY,                       # (T,) [mm]
        repose_ok=np.bool_(repose_fn is not None),
        depth_scale_ratio=SCALE_RATIO,                       # (T,) 奥行き補正の倍率
        key_frames=np.asarray(                               # 8d で採用したキーフレーム番号
            globals().get('KEY_FRAMES', np.arange(N_FRAMES)), np.int32),
        fulcrum_weights=np.asarray(                          # (T, P) 支点の接地の重み
            globals().get('FULCRUM_W', np.zeros((N_FRAMES, 0), np.float32)), np.float32),
        fulcrum_names=np.array(globals().get('FULCRUM_POINT_NAMES', []), dtype='<U16'),
        fps=np.float32(FPS),
        start_sec=np.float32(START_SEC),
        end_sec=np.float32(END_SEC),
        smooth_cutoff_hz=np.float32(SMOOTH_CUTOFF_HZ),
        root_cutoff_hz=np.float32(ROOT_CUTOFF_HZ),
        median_window=np.int32(MEDIAN_WINDOW),
        num_aug=np.int32(NUM_AUG),
        joint_names=np.array(SMPL_JOINT_NAMES),
        kintree_parents=SMPL_PARENTS,
        body_model=np.array(BODY_MODEL_NAME),
    )
    return MOTION_NPZ


save_motion()
print('モーションデータを保存しました:', MOTION_NPZ,
      f'({os.path.getsize(MOTION_NPZ) / 1024 ** 2:.1f} MB)')
print('共通の体型 betas:', np.round(betas_const, 3))

### 8b.（任意）SMPLFitter で全フレーム共通の体型に整える

SMPL 公式ファイルがある場合のみの**代替手段**です（既定はオフ）。
セル 8 で体型はすでに中央値 1 本に固定しているので通常は不要ですが、
[SMPLFitter](https://github.com/isarandi/smplfitter) の `share_beta=True` を使うと、
体型を共有したまま**頂点から SMPL パラメータを当てはめ直す**ことができます。
公式ファイルを持っていて、体型の推定をやり直したい場合に有効化してください。

In [ ]:
#@title 8b. 体型を共通化するリフィット（公式 SMPL がある場合のみ） { display-mode: "form" }
REFIT_SHARED_SHAPE = False  #@param {type:"boolean"}

if REFIT_SHARED_SHAPE and BODY_MODEL is not None:
    from smplfitter.pt import BodyFitter
    bm = BODY_MODEL.to(DEVICE)
    fitter = BodyFitter(bm).to(DEVICE)
    verts_t = torch.from_numpy(motion['vertices3d'] / 1000.0).float().to(DEVICE)
    joints_t = torch.from_numpy(motion['joints3d'] / 1000.0).float().to(DEVICE)
    fits = []
    # 体型はバッチ内で共有されるため、可能な限り 1 回で全フレームを流す
    # （チャンクに分けると境界ごとに体型が変わり、周期的な段差が出る）
    chunk = len(verts_t) if len(verts_t) <= 512 else 64
    with torch.inference_mode():
        for s in range(0, len(verts_t), chunk):
            fits.append(fitter.fit(target_vertices=verts_t[s:s + chunk],
                                   target_joints=joints_t[s:s + chunk],
                                   num_iter=3, beta_regularizer=1.0, share_beta=True,
                                   final_adjust_rots=True,
                                   requested_keys=['pose_rotvecs', 'shape_betas', 'trans']))
        pose_rotvecs = torch.cat([f['pose_rotvecs'] for f in fits])
        shape_betas = torch.cat([f['shape_betas'] for f in fits])
        trans = torch.cat([f['trans'] for f in fits])
        shape_betas = shape_betas.mean(0, keepdim=True).expand_as(shape_betas).contiguous()
        out = bm(pose_rotvecs=pose_rotvecs, shape_betas=shape_betas, trans=trans)
    motion['pose'] = pose_rotvecs.reshape(len(verts_t), -1, 3).cpu().numpy()
    motion['betas'] = shape_betas.cpu().numpy()
    motion['trans'] = trans.cpu().numpy()
    motion['vertices3d'] = lowpass(out['vertices'].cpu().numpy() * 1000.0, SMOOTH_CUTOFF_HZ, FPS)
    motion['joints3d'] = lowpass(out['joints'].cpu().numpy() * 1000.0, SMOOTH_CUTOFF_HZ, FPS)
    motion.pop('ground_offset_mm', None)   # 接地補正はやり直しになる
    save_motion()
    print('共通体型でリフィットしました。betas =', np.round(motion['betas'][0], 3))
else:
    print('スキップしました（公式 SMPL ファイルが無い、または REFIT_SHARED_SHAPE=False）。')

### 8c. 接地補正（足の滑りを取る）

単眼推定では**奥行きが最も不安定**で、さらに元動画のカメラの動きもそのまま全身の平行移動として出てしまいます。
その結果、足が地面をツルツル滑って見えます。ここでは

1. 足関節（足首・つま先）の高さから**床の高さ**を推定し、
2. 「**床に近く・水平方向にほとんど動いていない**」フレームを**接地**とみなし、
3. 接地している足が地面上で止まるように、**全身の位置**（平行移動）を補正します。

補正は平行移動だけなので、関節角度（ポーズ）には手を加えません。足の踏み替えは保たれます。

`ROOT_MOTION` で全身の移動の扱いを選べます。

| 値 | 動作 |
|---|---|
| `locked` | 低周波のドリフトだけを除去して**その場で踊る**。カメラのパンや奥行きドリフトが原因の滑りに最も効きます（既定） |
| `smoothed` | 移動は残しつつ水平方向の軌跡を 1 Hz で平滑化。接地補正の細かい成分も一部削れます |
| `full` | 接地補正後の移動をそのまま使います |

グラフの横軸は時間（フレーム）です。上段が足の床からの高さ（灰色の帯が接地と判定した区間）、
下段が足の水平速度で、**接地区間の速度が下がっていれば補正が効いています**。

In [ ]:
#@title 8c. 接地補正と全身の移動の扱い { display-mode: "form" }
import matplotlib.pyplot as plt

CONTACT_HEIGHT_MM = 80.0  #@param {type:"number"}
CONTACT_SPEED_MMPS = 350.0  #@param {type:"number"}
LOCK_LEAK_SEC = 5.0  #@param {type:"number"}
LOCK_CUTOFF_HZ = 0.3  #@param {type:"number"}

CONTACT_JOINTS = [7, 8, 10, 11]   # 左足首, 右足首, 左つま先, 右つま先
CONTACT_NAMES = ['L ankle', 'R ankle', 'L toe', 'R toe']


def smoothstep(x):
    x = np.clip(x, 0.0, 1.0)
    return x * x * (3.0 - 2.0 * x)


def horizontal_speed(p, fps):
    # 水平（x, z）方向の速さ [mm/s]。中央差分。
    v = np.empty(p.shape[:-1], np.float32)
    d = (p[2:] - p[:-2])[..., [0, 2]]
    v[1:-1] = np.linalg.norm(d, axis=-1) * (fps / 2.0)
    v[0], v[-1] = v[1], v[-2]
    return v


# 前回の補正を取り消してから計算する（セルを何度実行しても結果が同じになるように）
prev_off = motion.pop('ground_offset_mm', None)
if prev_off is not None:
    motion['vertices3d'] = motion['vertices3d'] - prev_off[:, None]
    motion['joints3d'] = motion['joints3d'] - prev_off[:, None]
    motion['trans'] = motion['trans'] - prev_off / 1000.0

if N_FRAMES < 12:
    print('フレーム数が少なすぎるため接地補正はスキップします。')
else:
    jnt = motion['joints3d']
    P = jnt[:, CONTACT_JOINTS]                       # (T, 4, 3) [mm]

    def slow(x, fc):
        # 遮断周波数が低いので、端で振れる Butterworth ではなくガウシアンを使う
        return gaussian_filter1d(np.asarray(x, np.float32),
                                 FPS / (2 * np.pi * max(fc, 1e-3)), axis=0, mode='nearest')

    # 接地点ごとの「地面に着いているときの高さ」を基準にする。
    # 足首はつま先より数 cm 高い位置にあるので、床を 1 枚の高さで代表させると
    # 足首がいつまでも「浮いている」と判定されてしまう。
    ref_y = np.percentile(P[..., 1], 75, axis=0)     # (4,) y は下向き = 大きいほど低い
    height = (ref_y[None] - P[..., 1]).astype(np.float32)   # 各点の基準面からの高さ [mm]
    eligible = (ref_y.max() - ref_y) < 150.0         # 一度も床付近に来ない点は除外
    speed = horizontal_speed(P, FPS)                 # (T, 4) [mm/s]

    # 速さは「絶対値」ではなく「そのフレームで一番遅い足との差」で見る。
    # カメラが動いている動画ではカメラ座標系で完全に止まる足は存在しないので、
    # 絶対値でしきい値を切ると接地が 1 つも検出できなくなる。
    v_floor = np.min(np.where(eligible[None], speed, np.inf), axis=1, keepdims=True)
    v_rel = speed - v_floor

    # 接地の重み（0〜1）。硬い 0/1 判定にすると接地の切り替わりで補正が飛ぶ。
    w = (smoothstep(1.0 - height / CONTACT_HEIGHT_MM)
         * smoothstep(1.0 - v_rel / CONTACT_SPEED_MMPS)
         # 一番遅い足まで速いフレーム（ジャンプ中など）は全部「非接地」にする
         * smoothstep(1.0 - v_floor / (4.0 * CONTACT_SPEED_MMPS)))
    # 一度も床付近に来ない点と、人物を検出できなかった（補間した）フレームは信用しない。
    # 重みが連続値なので、単発の誤検出はここで 0/1 に丸めず、そのまま小さい重みとして扱う
    # （前後 2 フレーム連続で接地していないと補正に効かないため、これで十分）。
    w = (w * eligible[None] * valid[:, None]).astype(np.float32)
    # --- 水平方向: 接地している足が動かないように全身をずらす ---
    corr = np.zeros((N_FRAMES, 2), np.float32)
    if FOOT_LOCK:
        # 前後フレームとも接地している点だけを使う。重みは 2 乗して、
        # 「確実に着いている点」を優先する（浮きかけのつま先に引っ張られないように）。
        W = (w[1:] * w[:-1]) ** 2
        den = W.sum(-1)
        step = (W[..., None] * (P[1:] - P[:-1])[..., [0, 2]]).sum(1) / np.maximum(den, 1e-6)[:, None]
        delta = -step * smoothstep(den / 0.5)[:, None]
        delta[den <= 1e-6] = 0.0
        # 漏れ項つきの積分。時定数 LOCK_LEAK_SEC 秒で元の軌跡に戻るので補正が暴走しない。
        lam = float(np.exp(-1.0 / max(LOCK_LEAK_SEC * FPS, 1e-6)))
        acc = np.zeros(2, np.float32)
        for t in range(1, N_FRAMES):
            acc = lam * acc + delta[t - 1]
            n = float(np.linalg.norm(acc))
            if n > 1000.0:
                acc = acc * (1000.0 / n)
            corr[t] = acc
        # 強くローパスすると、せっかく止めた接地区間の中で滑りが戻ってしまうので、
        # 高めの遮断周波数で軽くだけ均す（重み w が連続なので段差は出ない）
        corr = lowpass(corr, min(8.0, FPS / 2 * 0.9), FPS)

    # --- 垂直方向: 接地している足を基準面に戻す（ジャンプは潰さない） ---
    dy = np.zeros(N_FRAMES, np.float32)
    if FOOT_LOCK:
        wsum = w.sum(-1)
        # e = 接地している点が基準面からどれだけ浮いているか（上向きが正）
        e = (w * height).sum(-1) / np.maximum(wsum, 1e-6)
        gate = np.clip(gaussian_filter1d(np.clip(wsum, 0, 1), 2.0, mode='nearest'), 0, 1)
        # y は下向きが正なので、e だけ浮いていれば +e ずらせば基準面に戻る。
        # 滞空中は gate=0 なので補正は 0 になり、ジャンプの上下動はそのまま残る。
        dy = lowpass((e * gate)[:, None], min(3.0, ROOT_CUTOFF_HZ), FPS)[:, 0]
        # フィルタで滞空中ににじんだ分を gate で戻す。省くと踏み切りと着地が鈍る。
        dy = np.clip(dy * gate, -150.0, 150.0)

    # --- 全身の移動の扱い ---
    root_xz = jnt[:, 0][:, [0, 2]] + corr
    if ROOT_MOTION == 'locked':
        # 低周波のドリフト（カメラのパンや奥行きのずれ）だけを除去。
        # 全部消すと体重移動まで消えて、かえって足が滑って見える。
        off_xz = corr - (slow(root_xz, LOCK_CUTOFF_HZ) - root_xz.mean(0, keepdims=True))
    elif ROOT_MOTION == 'smoothed':
        off_xz = corr + (slow(root_xz, 1.0) - root_xz)
    else:
        off_xz = corr

    offset = np.stack([off_xz[:, 0], dy, off_xz[:, 1]], axis=1).astype(np.float32)
    motion['vertices3d'] = motion['vertices3d'] + offset[:, None]
    motion['joints3d'] = motion['joints3d'] + offset[:, None]
    motion['trans'] = motion['trans'] + offset / 1000.0
    motion['ground_offset_mm'] = offset

    # 床の高さ（足の裏 = 頂点ベース）を描画セルへ渡す
    contact_frames = np.flatnonzero(w.max(-1) > 0.5)
    if len(contact_frames) > 0:
        FLOOR_Y_MM = float(np.percentile(motion['vertices3d'][contact_frames][..., 1], 99.5))
    else:
        FLOOR_Y_MM = float(np.percentile(motion['vertices3d'][..., 1], 99.7))

    # --- 効果の確認 ---
    speed_after = horizontal_speed(motion['joints3d'][:, CONTACT_JOINTS], FPS)
    cm = w > 0.5
    print('接地率: ' + ' / '.join(
        f'{CONTACT_NAMES[i]} {100 * cm[:, i].mean():.0f}%' for i in range(4)))
    if cm.any():
        print(f'接地中の足の水平速度（中央値）: {np.median(speed[cm]):.0f} mm/s '
              f'-> {np.median(speed_after[cm]):.0f} mm/s')
    else:
        print('⚠️ 接地フレームが見つかりませんでした。CONTACT_HEIGHT_MM / '
              'CONTACT_SPEED_MMPS を大きくしてみてください。')
    print(f'補正量: 水平 最大 {np.abs(off_xz).max():.0f} mm / 垂直 平均 {np.abs(dy).mean():.0f} mm'
          f'（FOOT_LOCK={FOOT_LOCK}, ROOT_MOTION={ROOT_MOTION}）')
    save_motion()

    fig, axes = plt.subplots(2, 1, figsize=(9, 4.2), sharex=True)
    for i in range(4):
        axes[0].plot(height[:, i], lw=1, label=CONTACT_NAMES[i])
        axes[1].plot(speed[:, i], lw=0.8, alpha=0.35)
        axes[1].plot(speed_after[:, i], lw=1, label=CONTACT_NAMES[i])
    axes[0].fill_between(np.arange(N_FRAMES), 0, 1, where=cm.any(1),
                         transform=axes[0].get_xaxis_transform(), color='0.85', zorder=0)
    axes[0].axhline(CONTACT_HEIGHT_MM, color='r', lw=0.8, ls='--')
    axes[0].set_ylabel('height above plant [mm]')
    axes[0].set_title('shaded = detected contact | thin = before, thick = after', fontsize=9)
    axes[1].axhline(CONTACT_SPEED_MMPS, color='r', lw=0.8, ls='--')
    axes[1].set_ylabel('foot speed [mm/s]')
    axes[1].set_xlabel('frame')
    axes[1].set_ylim(0, max(CONTACT_SPEED_MMPS * 3, float(np.percentile(speed, 99))))
    axes[0].legend(fontsize=7, ncol=4)
    plt.tight_layout()
    plt.show()

### 8d. 支点クリーニングとキーフレーム化（Cascadeur 風）

[Cascadeur](https://cascadeur.com/) の *Fulcrum points* まわりのクリーンアップを、このノートブックの中で行います。
8c が**全身の平行移動だけ**で足滑りを直すのに対して、ここでは**関節角度（IK）とキーフレーム**まで手を入れます。

| Cascadeur の機能 | ここでの実装 |
|---|---|
| **Show fulcrum points** | 体を支えている点（足首・つま先・手首）を検出し、下のグラフに表示します。セル 11 の `SHOW_FULCRUM_MARKERS` を有効にすると、**プレビュー動画にもマーカー**（接地中は緑、浮きかけは橙）が出ます |
| **Using only key frames** | 選んだキーフレームだけを残して、間はすべて補間で作り直します。**キー以外のフレームに乗っていた推定ノイズ（細かな振動）がそのまま消えます** |
| **Prepare keys by fulcrums** | **支点の切り替わり（接地の開始・終了）フレームを必ずキーにします**。踏み込み・踏み切りの瞬間が鈍りません |
| **Adjust keys and interpolation** | キーの本数は「キーだけで補間しても全フレームが許容誤差に収まる」最小限を自動で選びます（`KEY_ANGLE_TOL_DEG` / `KEY_POS_TOL_MM` / `KEY_MAX_GAP`）。補間は回転が `RotationSpline`（C2 連続の 3 次回転スプライン）、位置が `PchipInterpolator`（行き過ぎない単調 3 次補間）。`KEY_INTERP='linear'` なら Slerp ＋ 線形 |
| **Fulcrum motion cleaning** | 支点が接地している区間では、その点が**空間に固定される**ように脚（腰→ひざ→足首）・腕（肩→ひじ→手首）を**2 ボーン IK** で解き直します。足先の向きは保ったまま角度だけ直すので、8c のように全身をずらさずに滑りが取れます |
| **Adjust Autoposing lock state** | `AUTOPOSE_LOCK` でどの部位を固定して解くかを選びます（`feet`＝足だけ / `feet_and_hands`＝手も支点にする（四つん這い・側転など） / `none`＝IK しない） |

処理の順番:

1. 支点の検出（接地の重み 0〜1）
2. 支点の切り替わりを必ずキーにして、キーフレームを選ぶ
3. キーだけを残して補間で作り直す（細かな振動が落ちる）
4. キーフレーム上で支点クリーニング（IK）→ 間をもう一度補間（キーだけの状態は保たれます）

**キーフレーム化を IK より先に行うのが大事です。** 先に足を止めてしまうと、フレームごとの推定ノイズまで
足先で打ち消すことになり、そのぶんが**ひざ・ひじの角度に押し込まれて逆に暴れます**。
同じ理由で、IK が直すのは `FULCRUM_FIX_HZ`（既定 2.5 Hz）以下のゆっくりしたずれ＝滑り・ドリフトだけにしてあります。
細かな振動のほうはキーフレーム化が落とします。

結果（振動の指標・接地中の足の速さ・キーの本数）はセルの出力に表示され、モーションは `motion.npz`
（`key_frames` と `fulcrum_weights` も一緒に保存）に書き戻されます。
**この後のプレビュー動画・FBX はすべてクリーンアップ後のモーション**を使います。
FBX も既定ではキーフレームだけを（3 次補間の指定つきで）書き出すので、Cascadeur 側でそのまま編集できます。


In [ ]:
#@title 8d. 支点クリーニングとキーフレーム化（Cascadeur 風） { display-mode: "form" }
CASC_CLEAN = True  #@param {type:"boolean"}
AUTOPOSE_LOCK = "feet_and_hands"  #@param ["feet", "feet_and_hands", "none"]
FULCRUM_FIX_HZ = 2.5  #@param {type:"number"}
FULCRUM_MAX_FIX_MM = 120.0  #@param {type:"number"}
KEEP_END_ORIENTATION = True  #@param {type:"boolean"}
USE_KEYFRAMES_ONLY = True  #@param {type:"boolean"}
KEY_ANGLE_TOL_DEG = 1.5  #@param {type:"number"}
KEY_POS_TOL_MM = 8.0  #@param {type:"number"}
KEY_MAX_GAP = 12  #@param {type:"integer"}
KEY_INTERP = "cubic"  #@param ["cubic", "linear"]

import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from scipy.spatial.transform import Slerp

try:
    from scipy.spatial.transform import RotationSpline   # C2 連続の 3 次回転スプライン
except Exception:
    RotationSpline = None

# 8c を飛ばしていても動くように、同じ道具・同じしきい値をここでも用意する
CONTACT_HEIGHT_MM = float(globals().get('CONTACT_HEIGHT_MM', 80.0))
CONTACT_SPEED_MMPS = float(globals().get('CONTACT_SPEED_MMPS', 350.0))
if 'smoothstep' not in globals():
    def smoothstep(x):
        x = np.clip(x, 0.0, 1.0)
        return x * x * (3.0 - 2.0 * x)
if 'horizontal_speed' not in globals():
    def horizontal_speed(p, fps):
        v = np.empty(p.shape[:-1], np.float32)
        v[1:-1] = np.linalg.norm((p[2:] - p[:-2])[..., [0, 2]], axis=-1) * (fps / 2.0)
        v[0], v[-1] = v[1], v[-2]
        return v

# 支点（fulcrum）= 体を支えている点。足首・つま先・手首を候補にする。
FULCRUM_POINT_IDS = [7, 10, 8, 11, 20, 21]
FULCRUM_POINT_NAMES = ['L ankle', 'L toe', 'R ankle', 'R toe', 'L wrist', 'R wrist']
FULCRUM_POINT_GROUP = np.array(['feet'] * 4 + ['hands'] * 2)
# IK を解く 2 ボーンの鎖: a=付け根 / b=中間 / c=先端（先端の位置を支点に合わせる）
FULCRUM_CHAINS = [
    dict(name='L leg', a=1, b=4, c=7, points=[7, 10], group='feet'),
    dict(name='R leg', a=2, b=5, c=8, points=[8, 11], group='feet'),
    dict(name='L arm', a=16, b=18, c=20, points=[20], group='hands'),
    dict(name='R arm', a=17, b=19, c=21, points=[21], group='hands'),
]
LOCK_GROUPS = {'feet': ('feet',), 'feet_and_hands': ('feet', 'hands'), 'none': ()}


def _unit(v, eps=1e-12):
    return v / np.maximum(np.linalg.norm(v, axis=-1, keepdims=True), eps)


def _angle_between(u, v):
    return np.arccos(np.clip((_unit(u) * _unit(v)).sum(-1), -1.0, 1.0))


def rot_axis_angle(axis, angle, fallback):
    # 軸が縮退しているフレーム（ひざが伸びきっている等）では fallback 軸を使う
    axis = np.where((np.linalg.norm(axis, axis=-1) > 1e-9)[:, None], axis, fallback)
    return Rotation.from_rotvec(_unit(axis) * np.asarray(angle, np.float64)[:, None]).as_matrix()


def rot_between(u, v):
    # u を v に向ける回転行列 (T, 3, 3)
    u, v = _unit(u), _unit(v)
    axis = np.cross(u, v)
    s = np.linalg.norm(axis, axis=-1)
    ang = np.where(s > 1e-9, np.arctan2(s, np.clip((u * v).sum(-1), -1.0, 1.0)), 0.0)
    axis = np.where((s > 1e-9)[:, None], axis, np.array([0.0, 0.0, 1.0]))
    return Rotation.from_rotvec(_unit(axis) * ang[:, None]).as_matrix()


def fk_global(pose, trans, rest):
    # 大域回転 G (T, J, 3, 3) と関節位置 P (T, J, 3)。単位は rest / trans と同じ [m]。
    T, J = pose.shape[:2]
    R = Rotation.from_rotvec(pose.reshape(-1, 3)).as_matrix().reshape(T, J, 3, 3)
    G, P = np.empty_like(R), np.empty((T, J, 3))
    for j, p in enumerate(SMPL_PARENTS):
        if p < 0:
            G[:, j], P[:, j] = R[:, j], rest[j] + trans
        else:
            G[:, j] = G[:, p] @ R[:, j]
            P[:, j] = P[:, p] + np.einsum('tij,j->ti', G[:, p], rest[j] - rest[p])
    return G, P


def fulcrum_weights(joints_mm, groups_on):
    # 各支点候補が「接地している度合い」0〜1 を返す (T, 6)。判定の考え方は 8c と同じ。
    P = joints_mm[:, FULCRUM_POINT_IDS]
    is_foot = FULCRUM_POINT_GROUP == 'feet'
    ref = np.percentile(P[..., 1], 75, axis=0)          # 足はその点自身の接地高さを基準に
    floor = float(ref[is_foot].max())                   # 手は足から推定した床を基準に
    ref = np.where(is_foot, ref, floor)
    height = (ref[None] - P[..., 1]).astype(np.float32)
    speed = horizontal_speed(P, FPS)
    eligible = np.isin(FULCRUM_POINT_GROUP, list(groups_on))
    eligible &= np.where(is_foot,
                         (ref[is_foot].max() - ref) < 150.0,      # 一度も床付近に来ない足は除外
                         height.min(0) < CONTACT_HEIGHT_MM)       # 手は床に触れたときだけ支点
    ref_pts = eligible & is_foot
    if not ref_pts.any():
        ref_pts = eligible if eligible.any() else np.ones(len(eligible), bool)
    # 速さは絶対値ではなく「一番遅い足との差」で見る（カメラが動いていても検出できるように）
    v_floor = np.min(np.where(ref_pts[None], speed, np.inf), axis=1, keepdims=True)
    w = (smoothstep(1.0 - height / CONTACT_HEIGHT_MM)
         * smoothstep(1.0 - (speed - v_floor) / CONTACT_SPEED_MMPS)
         * smoothstep(1.0 - v_floor / (4.0 * CONTACT_SPEED_MMPS)))
    return (w * eligible[None] * valid[:, None]).astype(np.float32)


def anchor_delta(P_mm, w, thr=0.3, max_fix=120.0, fix_hz=2.5):
    # 接地している区間ごとに「その点が居るべき 1 点（アンカー）」を決め、そこへのずれを返す [mm]
    d = np.zeros_like(P_mm)
    on = (w > thr).astype(np.int8)
    edges = np.flatnonzero(np.diff(np.concatenate([[0], on, [0]])))
    for s, e in zip(edges[::2], edges[1::2]):
        ww = w[s:e, None]
        d[s:e] = (P_mm[s:e] * ww).sum(0) / max(float(ww.sum()), 1e-6) - P_mm[s:e]
    # IK で直すのは「ゆっくりしたずれ（滑り・ドリフト）」だけにする。
    # 細かな振動まで足先で打ち消そうとすると、そのぶんがひざ・ひじの角度に押し込まれて
    # かえって暴れる。振動のほうはキーフレーム化で落とす。
    sigma = FPS / (2 * np.pi * max(float(fix_hz), 1e-3))
    if sigma > 0.3:
        d = gaussian_filter1d(d, sigma, axis=0, mode='nearest')
    d = d * w[:, None]
    n = np.linalg.norm(d, axis=-1, keepdims=True)
    return d * np.minimum(1.0, max_fix / np.maximum(n, 1e-9))


def two_bone_ik(G, P, rest, a, b, c, target):
    # 先端 c が target に来るように、b の曲げ角 -> a の向き の順に解く（標準的な 2 ボーン IK）
    pa, pb, pc = P[:, a], P[:, b], P[:, c]
    l1 = float(np.linalg.norm(rest[b] - rest[a]))
    l2 = float(np.linalg.norm(rest[c] - rest[b]))
    dist = np.clip(np.linalg.norm(target - pa, axis=-1), abs(l1 - l2) + 1e-4, l1 + l2 - 1e-4)
    beta0 = _angle_between(pa - pb, pc - pb)
    beta1 = np.arccos(np.clip((l1 ** 2 + l2 ** 2 - dist ** 2) / (2 * l1 * l2), -1.0, 1.0))
    fallback = np.einsum('tij,j->ti', G[:, b], np.array([1.0, 0.0, 0.0]))
    Rb = rot_axis_angle(np.cross(pa - pb, pc - pb), beta1 - beta0, fallback)
    pc1 = pb + np.einsum('tij,tj->ti', Rb, pc - pb)
    R3 = rot_between(pc1 - pa, target - pa)
    return R3 @ G[:, a], R3 @ Rb @ G[:, b]


def fulcrum_clean(pose, trans, rest, groups_on, joints_mm=None, frames=None):
    # 支点が接地している区間で、その点が空間に止まるよう IK で関節角を直す。
    # frames を渡すと、そのフレームだけを直す（キーフレームだけ締め直したいとき）。
    G, P = fk_global(pose.astype(np.float64), trans.astype(np.float64), rest)
    P_mm = P * 1000.0 if joints_mm is None else joints_mm
    w = fulcrum_weights(P_mm, groups_on)
    out = pose.astype(np.float64).copy()
    sel = np.ones(len(pose), bool)
    if frames is not None:
        sel[:] = False
        sel[np.asarray(frames, int)] = True
    for ch in FULCRUM_CHAINS:
        if ch['group'] not in groups_on:
            continue
        cols = [FULCRUM_POINT_IDS.index(j) for j in ch['points']]
        ws = w[:, cols]
        den = ws.sum(1, keepdims=True)
        # 足首とつま先が両方接地していれば、両方のずれの重み付き平均を足首に与える
        # （足の向きは変えないので、足全体が剛体として同じだけ動く）
        delta = sum(ws[:, k:k + 1] * anchor_delta(P_mm[:, j], ws[:, k], max_fix=FULCRUM_MAX_FIX_MM,
                                 fix_hz=FULCRUM_FIX_HZ)
                    for k, j in enumerate(ch['points'])) / np.maximum(den, 1e-6)
        delta = np.where((den > 1e-4) & sel[:, None], delta, 0.0)
        target = P[:, ch['c']] + delta / 1000.0
        Ga, Gb = two_bone_ik(G, P, rest, ch['a'], ch['b'], ch['c'], target)
        par = int(SMPL_PARENTS[ch['a']])
        # 直す必要が無いフレームには触らない。ひざ・ひじが伸びきっていると曲げ軸が決まらず、
        # 目標が現在位置と同じでも数値誤差で 1 度ほど動いてしまうため。
        m = np.linalg.norm(delta, axis=-1) > 0.1      # [mm]
        if not m.any():
            continue
        out[m, ch['a']] = Rotation.from_matrix(
            np.einsum('tji,tjk->tik', G[m, par], Ga[m])).as_rotvec()
        out[m, ch['b']] = Rotation.from_matrix(
            np.einsum('tji,tjk->tik', Ga[m], Gb[m])).as_rotvec()
        if KEEP_END_ORIENTATION:
            # 先端（足・手）の大域的な向きは変えない。足裏が浮いたり捻れたりしないように。
            out[m, ch['c']] = Rotation.from_matrix(
                np.einsum('tji,tjk->tik', Gb[m], G[m, ch['c']])).as_rotvec()
    return out.astype(np.float32), w


def continuous_quats(pose):
    # 四元数は q と -q が同じ回転なので、時間方向に符号をそろえてから補間誤差を測る
    q = Rotation.from_rotvec(pose.reshape(-1, 3)).as_quat().reshape(pose.shape[0], -1, 4)
    d = (q[1:] * q[:-1]).sum(-1)
    q[1:] *= np.cumprod(np.where(d < 0, -1.0, 1.0), axis=0)[..., None]
    return q


def select_keyframes(pose, trans_mm, must_keys, tol_deg, tol_mm, max_gap):
    # 「キーだけ残して補間しても、どのフレームも許容誤差に収まる」最小限のキーを選ぶ
    # （曲線簡略化と同じ再帰分割。must_keys = 支点が切り替わるフレームで、必ずキーにする）
    q = continuous_quats(pose)
    T = len(pose)
    keys = sorted(set([0, T - 1]) | {int(k) for k in must_keys if 0 <= k < T})
    result = set(keys)
    stack = list(zip(keys[:-1], keys[1:]))
    while stack:
        i, j = stack.pop()
        if j - i <= 1:
            continue
        k = np.arange(i + 1, j)
        t = ((k - i) / (j - i))[:, None]
        qq = _unit(q[i][None] * (1 - t[..., None]) + q[j][None] * t[..., None])
        e_rot = np.degrees(2 * np.arccos(np.clip(np.abs((qq * q[i + 1:j]).sum(-1)), 0, 1))).max(1)
        e_pos = np.linalg.norm(
            trans_mm[i][None] * (1 - t) + trans_mm[j][None] * t - trans_mm[i + 1:j], axis=-1)
        score = np.maximum(e_rot / max(tol_deg, 1e-6), e_pos / max(tol_mm, 1e-6))
        if score.max() > 1.0:
            s = int(k[np.argmax(score)])
        elif j - i > max_gap:
            s = (i + j) // 2
        else:
            continue
        result.add(s)
        stack += [(i, s), (s, j)]
    return np.array(sorted(result), np.int32)


def rebuild_from_keys(pose, trans, keys, mode):
    # キーフレームだけを使ってモーションを作り直す（キー以外に乗っていたノイズが消える）
    x = np.arange(len(pose))
    out = np.empty_like(pose, np.float64)
    cubic = (mode == 'cubic') and len(keys) >= 3
    for j in range(pose.shape[1]):
        Rk = Rotation.from_rotvec(pose[keys, j].astype(np.float64))
        if cubic and RotationSpline is not None:
            out[:, j] = RotationSpline(keys, Rk)(x).as_rotvec()
        else:
            out[:, j] = Slerp(keys, Rk)(x).as_rotvec()
    if cubic:
        # 位置は行き過ぎない（オーバーシュートしない）単調 3 次補間。足を踏み込む瞬間が崩れない。
        tr = PchipInterpolator(keys, trans[keys].astype(np.float64), axis=0)(x)
    else:
        tr = np.stack([np.interp(x, keys, trans[keys, k]) for k in range(3)], axis=1)
    return out.astype(np.float32), tr.astype(np.float32)


def clean_runs(on, min_run):
    # 1〜2 フレームだけの接地・離地は判定のばたつきなので無視する
    on = np.asarray(on, bool).copy()
    for val in (True, False):
        idx = np.flatnonzero(np.diff(np.concatenate([[False], on == val, [False]]).astype(np.int8)))
        for s, e in zip(idx[::2], idx[1::2]):
            if e - s < min_run:
                on[s:e] = not val
    return on


def contact_switch_frames(w, thr=0.3, min_run=2):
    # 支点が入れ替わるフレーム（接地の開始・終了）。ここは必ずキーにする。
    fr = set()
    for i in range(w.shape[1]):
        on = clean_runs(w[:, i] > thr, min_run).astype(np.int8)
        e = np.flatnonzero(np.diff(on))
        fr |= set(e.tolist()) | set((e + 1).tolist())
    return sorted(fr)


LOCK_ON = LOCK_GROUPS[AUTOPOSE_LOCK]
KEY_FRAMES = np.arange(N_FRAMES, dtype=np.int32)
FULCRUM_W = np.zeros((N_FRAMES, len(FULCRUM_POINT_IDS)), np.float32)

if not CASC_CLEAN:
    print('スキップしました（CASC_CLEAN=False）。')
elif repose_fn is None:
    print('⚠️ 体モデルで再ポーズできない環境なので、このクリーンアップはスキップします。')
elif N_FRAMES < 8:
    print('フレーム数が少なすぎるためスキップします。')
else:
    # 静止姿勢の関節位置 [m]（骨の長さ = セル 8 で固定した共通体型）
    _, j_rest_mm = repose(np.zeros((1, N_JOINTS, 3), np.float32), motion['betas'][:1],
                          np.zeros((1, 3), np.float32), repose_fn)
    J_REST = (j_rest_mm[0] / 1000.0).astype(np.float64)

    pose0 = motion['pose'].astype(np.float32)
    trans0 = motion['trans'].astype(np.float32)
    joints0 = motion['joints3d'].copy()
    _, P_chk = fk_global(pose0.astype(np.float64), trans0.astype(np.float64), J_REST)
    fk_err = float(np.abs(P_chk * 1000.0 - joints0).max())
    print(f'順運動学の自己検証: motion.npz の関節との最大差 {fk_err:.2f} mm')

    if fk_err > 50.0:
        print('⚠️ 差が大きいのでクリーンアップは行いません（8b のリフィット直後などに起こります）。')
    else:
        speed_before = horizontal_speed(joints0[:, FULCRUM_POINT_IDS], FPS)
        jitter_before = jitter_metric(joints0)

        # ---- 1. 支点の検出 ----
        w1 = fulcrum_weights(joints0, LOCK_ON or ('feet',))
        print('支点: ' + ' / '.join(
            f'{FULCRUM_POINT_NAMES[i]} {100 * (w1[:, i] > 0.5).mean():.0f}%'
            for i in range(len(FULCRUM_POINT_NAMES))) + '（接地率）')

        # ---- 2. 支点でキーを用意 → キーフレームだけ残して作り直す ----
        # キーフレーム化を先にやるのが大事。先に IK で足を止めてしまうと、フレームごとの
        # 推定ノイズまで足先で打ち消すことになり、そのぶんがひざ・ひじの角度に押し込まれる。
        pose1 = pose0
        pose2, trans2 = pose1, trans0
        if USE_KEYFRAMES_ONLY:
            must = contact_switch_frames(w1, min_run=max(2, int(round(0.08 * FPS))))
            must += np.flatnonzero(np.diff(valid.astype(np.int8)) != 0).tolist()
            KEY_FRAMES = select_keyframes(pose1, trans0 * 1000.0, must,
                                          KEY_ANGLE_TOL_DEG, KEY_POS_TOL_MM, max(2, KEY_MAX_GAP))
            pose2, trans2 = rebuild_from_keys(pose1, trans0, KEY_FRAMES, KEY_INTERP)
            print(f'キーフレーム化: {N_FRAMES} フレーム -> キー {len(KEY_FRAMES)} 本 '
                  f'({100 * len(KEY_FRAMES) / N_FRAMES:.0f}%、うち支点の切り替わり '
                  f'{len(set(must))} 本) / 補間 {KEY_INTERP}')

        # ---- 3. 支点クリーニング: 接地している支点が空間に止まるよう IK で解き直す ----
        pose3 = pose2
        if LOCK_ON:
            _, P2 = fk_global(pose2.astype(np.float64), trans2.astype(np.float64), J_REST)
            pose3, _ = fulcrum_clean(pose2, trans2, J_REST, LOCK_ON, joints_mm=P2 * 1000.0,
                                     frames=KEY_FRAMES if USE_KEYFRAMES_ONLY else None)
            if USE_KEYFRAMES_ONLY:
                # 直したのはキーだけなので、間はもう一度補間で作り直す（キーだけの状態を保つ）
                pose3, trans2 = rebuild_from_keys(pose3, trans2, KEY_FRAMES, KEY_INTERP)
            print(f'支点クリーニング: {AUTOPOSE_LOCK} を固定して、'
                  + ('キーフレーム上で ' if USE_KEYFRAMES_ONLY else '全フレームで ')
                  + f'IK で解き直しました（{FULCRUM_FIX_HZ:g} Hz 以下のずれを補正）。')
        else:
            print('支点クリーニングはスキップしました（AUTOPOSE_LOCK="none"）。')

        motion['pose'] = pose3.astype(np.float32)
        motion['trans'] = trans2.astype(np.float32)
        motion['vertices3d'], motion['joints3d'] = repose(
            motion['pose'], motion['betas'], motion['trans'], repose_fn)
        FULCRUM_W = fulcrum_weights(motion['joints3d'], LOCK_ON or ('feet',))

        # ---- 効果の確認と保存 ----
        speed_after = horizontal_speed(motion['joints3d'][:, FULCRUM_POINT_IDS], FPS)
        cm = (w1 > 0.5) | (FULCRUM_W > 0.5)
        print(f'細かな振動（関節の 2 階差分）: {jitter_before:.2f} mm -> '
              f'{jitter_metric(motion["joints3d"]):.2f} mm')
        if cm.any():
            print(f'支点が接地している間の水平速度（中央値）: {np.median(speed_before[cm]):.0f} mm/s '
                  f'-> {np.median(speed_after[cm]):.0f} mm/s')
        print(f'関節位置の変化量（最大）: {np.abs(motion["joints3d"] - joints0).max():.0f} mm')

        # 床の高さを支点から取り直して、描画セル・FBX に渡す
        cf = np.flatnonzero(FULCRUM_W.max(-1) > 0.5)
        FLOOR_Y_MM = float(np.percentile(motion['vertices3d'][cf][..., 1], 99.5) if len(cf)
                           else np.percentile(motion['vertices3d'][..., 1], 99.7))
        save_motion()

        # ---- Show fulcrum points: 支点の時系列 ----
        fig, axes = plt.subplots(2, 1, figsize=(9, 4.6), sharex=True)
        for i, nm in enumerate(FULCRUM_POINT_NAMES):
            axes[0].plot(FULCRUM_W[:, i], lw=1.0, label=nm)
            axes[1].plot(speed_before[:, i], lw=0.7, alpha=0.3)
            axes[1].plot(speed_after[:, i], lw=1.0, label=nm)
        axes[0].fill_between(np.arange(N_FRAMES), 0, 1, where=(FULCRUM_W > 0.5).any(1),
                             transform=axes[0].get_xaxis_transform(), color='0.88', zorder=0)
        if USE_KEYFRAMES_ONLY and len(KEY_FRAMES) < N_FRAMES:
            axes[0].plot(KEY_FRAMES, np.full(len(KEY_FRAMES), 1.08), '|', ms=6, color='k')
        axes[0].set_ylabel('fulcrum weight')
        axes[0].set_ylim(-0.05, 1.18)
        axes[0].set_title('fulcrum points (shaded = supported) | top ticks = key frames',
                          fontsize=9)
        axes[1].set_ylabel('speed [mm/s]')
        axes[1].set_xlabel('frame')
        axes[1].set_title('thin = before 8d, thick = after 8d', fontsize=9)
        axes[1].set_ylim(0, max(CONTACT_SPEED_MMPS * 3, float(np.percentile(speed_before, 99))))
        axes[0].legend(fontsize=7, ncol=6)
        plt.tight_layout()
        plt.show()


---
## 9. 抽出結果の確認（元フレームへの重ね描画）

推定した 3D 頂点を元のフレームに投影して重ねます。人物にきれいに重なっていれば抽出は成功です。
ずれている場合は、区間を変える・`PERSON_SELECT` を変える・`MAX_HEIGHT` を上げる、などを試してください。

※ 8c の接地補正は体の位置を意図的にずらすため、この重ね描画では**補正前の位置**に戻して表示しています。
（8d の支点クリーニングは関節角度そのものを直すので、そのぶんは戻せません。足元が数 cm ずれて見えることがあります。）

In [ ]:
#@title 9. 重ね描画でチェック { display-mode: "form" }
import matplotlib.pyplot as plt


def intrinsics_from_fov(fov_degrees, imshape):
    # NLF が既定で仮定しているカメラ（対角ではなく長辺基準の画角 55 度）と同じ式
    h, w = imshape[:2]
    f = float(max(h, w)) / (2.0 * np.tan(np.deg2rad(fov_degrees) / 2.0))
    return np.array([[f, 0, w / 2.0], [0, f, h / 2.0], [0, 0, 1]], np.float32)


def project(points_cam, K):
    z = np.maximum(points_cam[..., 2:], 1e-3)
    uv = points_cam[..., :2] / z
    return uv * np.array([K[0, 0], K[1, 1]], np.float32) + np.array([K[0, 2], K[1, 2]], np.float32)


SEG_W, SEG_H = SEG_INFO['size']
K_ORIG = intrinsics_from_fov(55.0, (SEG_H, SEG_W))

# 接地補正は体を「元の見え方」から意図的にずらすので、重ね描画では取り消して表示する
cam_verts = motion['vertices3d'] - motion.get(
    'ground_offset_mm', np.zeros((N_FRAMES, 3), np.float32))[:, None]

keys = sorted(k for k in preview if k < N_FRAMES)
if keys:
    fig, axes = plt.subplots(1, len(keys), figsize=(4 * len(keys), 4 * SEG_H / max(SEG_W, 1)))
    axes = np.atleast_1d(axes)
    for ax, k in zip(axes, keys):
        uv = project(cam_verts[k], K_ORIG)
        ax.imshow(preview[k])
        ax.scatter(uv[::12, 0], uv[::12, 1], s=1.0, c='lime', alpha=0.45)
        ax.set_title(f'frame {k}' + ('' if valid[k] else ' (interpolated)'), fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('プレビュー用のフレームがありません。')

---
## 10. マネキンのメッシュを組み立てる

**方式 A: SMPL メッシュ**（公式ファイルがある場合）
: SMPL の面情報（13,776 三角形）をそのまま使い、推定された頂点をフレームごとに差し替えます。

**方式 B: パーツ凸包マネキン**（既定 / 公式ファイル不要）
: SMPL の 6,890 頂点を体のパーツ（関節）ごとに分け、**パーツごとの凸包**を取って
  デッサン人形のような多面体マネキンを作ります。
  頂点の所属パーツは、TorchScript モデルの中に入っているスキニングウェイト（LBS weights）から取得し、
  取れない場合は「最も近い関節」で代用します。
  凸包の面の構成（トポロジ）は 1 フレームだけで計算し、以降のフレームは同じ面構成のまま頂点を差し替えるので高速です。

In [ ]:
#@title 10. マネキンのメッシュを作る { display-mode: "form" }
from scipy.spatial import ConvexHull


def try_get_lbs_weights(model, model_name, n_verts):
    # NLF の TorchScript には SMPL のバッファが入っているので、そこからスキニングウェイトを拝借する
    try:
        bm = getattr(model.body_models, model_name)
        w = bm.weights.detach().float().cpu().numpy()
        if w.ndim == 2 and w.shape[0] == n_verts:
            return w
    except Exception:
        pass
    return None


def vertex_part_labels(verts_ref, joints_ref, lbs_weights=None):
    if lbs_weights is not None:
        return lbs_weights.argmax(-1).astype(np.int32)
    d = np.linalg.norm(verts_ref[:, None, :] - joints_ref[None, :, :], axis=-1)
    return d.argmin(1).astype(np.int32)


def build_part_hull_topology(verts_ref, labels, num_parts, min_points=8):
    # パーツごとの凸包 -> 「頂点インデックス列 + 固定の面」に変換する
    idx_chunks, face_chunks, face_part = [], [], []
    offset = 0
    for p in range(num_parts):
        member = np.flatnonzero(labels == p)
        if len(member) < min_points:
            continue
        pts = verts_ref[member]
        try:
            hull = ConvexHull(pts, qhull_options='QJ')
        except Exception:
            continue
        used = np.unique(hull.simplices)
        remap = np.full(len(member), -1, np.int64)
        remap[used] = np.arange(len(used))
        tris = remap[hull.simplices]
        v = pts[used]
        a, b, c = v[tris[:, 0]], v[tris[:, 1]], v[tris[:, 2]]
        n = np.cross(b - a, c - a)
        flip = np.einsum('ij,ij->i', n, (a + b + c) / 3.0 - v.mean(0)) < 0
        tris[flip] = tris[flip][:, ::-1]
        idx_chunks.append(member[used])
        face_chunks.append(tris + offset)
        face_part.append(np.full(len(tris), p, np.int32))
        offset += len(used)
    return (np.concatenate(idx_chunks), np.concatenate(face_chunks).astype(np.int32),
            np.concatenate(face_part))


VERTS = motion['vertices3d']                 # (T, V, 3) [mm]
JOINTS = motion['joints3d']                  # (T, J, 3) [mm]
ref = int(np.flatnonzero(valid)[len(np.flatnonzero(valid)) // 2])  # 代表フレーム

use_smpl_mesh = (MANNEQUIN_STYLE == 'smpl_mesh'
                 or (MANNEQUIN_STYLE == 'auto' and SMPL_FACES is not None))
if use_smpl_mesh and SMPL_FACES is None:
    print('⚠️ SMPL 公式ファイルが無いのでパーツ凸包マネキンに切り替えます。')
    use_smpl_mesh = False

if use_smpl_mesh:
    VERTEX_MAP = np.arange(VERTS.shape[1])
    FACES = SMPL_FACES
    print(f'SMPL メッシュを使用します: 頂点 {len(VERTEX_MAP)} / 面 {len(FACES)}')
else:
    lbs = try_get_lbs_weights(nlf_model, BODY_MODEL_NAME, VERTS.shape[1])
    print('スキニングウェイト:', 'TorchScript から取得' if lbs is not None else '最近傍関節で代用')
    LABELS = vertex_part_labels(VERTS[ref], JOINTS[ref], lbs)
    VERTEX_MAP, FACES, FACE_PART = build_part_hull_topology(
        VERTS[ref], LABELS, num_parts=JOINTS.shape[1])
    print(f'パーツ凸包マネキン: パーツ {len(np.unique(FACE_PART))} / '
          f'頂点 {len(VERTEX_MAP)} / 面 {len(FACES)}')

MESH_VERTS = VERTS[:, VERTEX_MAP] / 1000.0   # (T, M, 3) [m]
print('マネキンの頂点列:', MESH_VERTS.shape)

---
## 11. レンダリング

カメラ座標系（x=右 / y=下 / z=奥）のまま描画します。

* `CAMERA_MODE='fit'` … 元動画と同じ視点のまま、シーケンス全体が画面に収まるよう焦点距離と中心を自動調整
* `CAMERA_MODE='original'` … 元動画とまったく同じ画角（人物の位置もそのまま）
* `VIEW_AZIMUTH_DEG` … 縦軸まわりに回転させて別アングルから撮影
* `SHOW_FLOOR` … 足元の最下点に市松模様の床を敷きます（奥行きが分かりやすくなります）

レンダラは OpenGL 不要の**ソフトウェアレンダラ**（三角形を奥から順に塗る画家アルゴリズム）です。
Colab でも追加インストール無しで確実に動きます。`pyrender` が使える環境なら `RENDER_BACKEND='pyrender'`
にすると、より陰影のきれいな描画になります（失敗したら自動でソフトウェアに戻ります）。

In [ ]:
#@title 11. マネキン動画をレンダリング { display-mode: "form" }
RENDER_BACKEND = "software"  #@param ["software", "pyrender"]
BODY_COLOR = "#D8D2C6"  #@param {type:"string"}
BG_COLOR = "#1C2029"  #@param {type:"string"}
SHOW_FULCRUM_MARKERS = True  #@param {type:"boolean"}

from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.collections import PolyCollection
from matplotlib.figure import Figure


def hex2rgb(s):
    s = s.lstrip('#')
    return np.array([int(s[i:i + 2], 16) / 255.0 for i in (0, 2, 4)], np.float32)


def rotate_about_y(points, center, degrees):
    th = np.deg2rad(degrees)
    c, s = np.cos(th), np.sin(th)
    R = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]], np.float32)  # y 軸（上下）まわり
    return (points - center) @ R.T + center


def fit_camera(points, imshape, margin=0.14):
    h, w = imshape[:2]
    p = points.reshape(-1, 3)
    p = p[p[:, 2] > 1e-3]
    u, v = p[:, 0] / p[:, 2], p[:, 1] / p[:, 2]
    u0, u1 = np.percentile(u, 0.1), np.percentile(u, 99.9)
    v0, v1 = np.percentile(v, 0.1), np.percentile(v, 99.9)
    f = min(w / max(u1 - u0, 1e-6), h / max(v1 - v0, 1e-6)) * (1.0 - margin)
    return np.array([[f, 0, w / 2 - f * (u0 + u1) / 2],
                     [0, f, h / 2 - f * (v0 + v1) / 2], [0, 0, 1]], np.float32)


def make_floor_grid(center_xz, y_level, half_size, n_cells=14):
    xs = np.linspace(center_xz[0] - half_size, center_xz[0] + half_size, n_cells + 1)
    zs = np.linspace(center_xz[1] - half_size, center_xz[1] + half_size, n_cells + 1)
    verts, faces, shade = [], [], []
    for i in range(n_cells):
        for j in range(n_cells):
            o = len(verts)
            verts += [[xs[i], y_level, zs[j]], [xs[i + 1], y_level, zs[j]],
                      [xs[i + 1], y_level, zs[j + 1]], [xs[i], y_level, zs[j + 1]]]
            faces += [[o, o + 1, o + 2], [o, o + 2, o + 3]]
            c = 0.80 if (i + j) % 2 == 0 else 0.66
            shade += [c, c]
    return (np.array(verts, np.float32), np.array(faces, np.int32),
            np.array(shade, np.float32)[:, None] * np.ones(3, np.float32))


def shade_faces(verts, faces, base_color, light_dir=(0.35, -0.75, -0.55),
                ambient=0.42, diffuse=0.72):
    a, b, c = verts[faces[:, 0]], verts[faces[:, 1]], verts[faces[:, 2]]
    n = np.cross(b - a, c - a)
    n = n / np.maximum(np.linalg.norm(n, axis=-1, keepdims=True), 1e-9)
    l = np.asarray(light_dir, np.float32)
    l = l / np.linalg.norm(l)
    lam = np.abs(n @ l)
    return np.clip(np.asarray(base_color, np.float32)[None] * (ambient + diffuse * lam)[:, None],
                   0, 1)


def render_software(verts, faces, K, imshape, body_color, bg_color, extra=None):
    h, w = imshape[:2]
    V, F, C = [verts], [faces], [shade_faces(verts, faces, body_color)]
    n_body_faces = len(faces)
    if extra is not None:
        ev, ef, ec = extra
        V.append(ev)
        F.append(ef + len(verts))
        C.append(np.clip(shade_faces(ev, ef, (1.0, 1.0, 1.0)) * ec, 0, 1))
    V, F, C = np.concatenate(V), np.concatenate(F), np.concatenate(C)
    tri = V[F]
    depth = tri[..., 2].mean(1)
    n = np.cross(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])
    facing = np.einsum('ij,ij->i', n, tri.mean(1)) < 0   # カメラ（原点）を向いている面だけ描く
    facing[n_body_faces:] = True                        # 床は両面描画
    keep = (depth > 1e-3) & facing
    F, C, depth = F[keep], C[keep], depth[keep]
    order = np.argsort(-depth)                          # 奥から手前へ
    polys = project(V, K)[F[order]]

    fig = Figure(figsize=(w / 100.0, h / 100.0), dpi=100)
    canvas = FigureCanvasAgg(fig)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.axis('off')
    fig.patch.set_facecolor(bg_color)
    ax.set_facecolor(bg_color)
    ax.add_collection(PolyCollection(polys, facecolors=C[order], edgecolors='none'))
    canvas.draw()
    return np.asarray(canvas.buffer_rgba())[..., :3].copy()


class PyrenderBackend:
    def __init__(self, imshape, bg_color):
        os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
        import pyrender
        import trimesh
        self.pyrender, self.trimesh = pyrender, trimesh
        self.renderer = pyrender.OffscreenRenderer(imshape[1], imshape[0])
        self.bg = bg_color

    def render(self, verts, faces, K, imshape, body_color, bg_color, extra=None):
        pyrender, trimesh = self.pyrender, self.trimesh
        scene = pyrender.Scene(bg_color=[*bg_color, 1.0], ambient_light=(0.45, 0.45, 0.45))
        flip = np.array([1, -1, -1], np.float32)   # OpenCV 座標系 -> OpenGL 座標系
        mat = pyrender.MetallicRoughnessMaterial(
            metallicFactor=0.15, roughnessFactor=0.7, alphaMode='OPAQUE',
            baseColorFactor=[*body_color, 1.0], doubleSided=True)
        scene.add(pyrender.Mesh.from_trimesh(trimesh.Trimesh(verts * flip, faces), material=mat))
        if extra is not None:
            ev, ef, ec = extra
            m = trimesh.Trimesh(ev * flip, ef, process=False)
            m.visual.face_colors = np.concatenate(
                [np.clip(ec, 0, 1), np.ones((len(ec), 1), np.float32)], axis=1)
            scene.add(pyrender.Mesh.from_trimesh(m, smooth=False))
        cam = pyrender.IntrinsicsCamera(fx=float(K[0, 0]), fy=float(K[1, 1]),
                                        cx=float(K[0, 2]), cy=float(K[1, 2]),
                                        znear=0.05, zfar=100.0)
        scene.add(cam, pose=np.eye(4))
        for pose in _raymond_light_poses():
            scene.add(pyrender.DirectionalLight(color=np.ones(3), intensity=2.0), pose=pose)
        color, _ = self.renderer.render(scene)
        return np.asarray(color)[..., :3]


def _raymond_light_poses():
    poses = []
    for phi in [0.0, 2 * np.pi / 3, 4 * np.pi / 3]:
        theta = np.pi / 6
        z = np.array([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)])
        z /= np.linalg.norm(z)
        x = np.array([-z[1], z[0], 0.0])
        x = x / np.linalg.norm(x) if np.linalg.norm(x) > 0 else np.array([1.0, 0.0, 0.0])
        m = np.eye(4)
        m[:3, :3] = np.c_[x, np.cross(z, x), z]
        poses.append(m)
    return poses


# ---- 支点（fulcrum points）のマーカー ----
# 8d が検出した「体を支えている点」を、接地しているフレームだけ小さな八面体で描く。
OCTA_V = np.array([[1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, 0, 1], [0, 0, -1]], np.float32)
OCTA_F = np.array([[0, 2, 4], [2, 1, 4], [1, 3, 4], [3, 0, 4],
                   [2, 0, 5], [1, 2, 5], [3, 1, 5], [0, 3, 5]], np.int32)
MARKER_ON = np.array([0.35, 0.95, 0.45], np.float32)    # 接地中
MARKER_OFF = np.array([0.95, 0.75, 0.30], np.float32)   # 浮きかけ


def merge_extra(*parts):
    # (頂点, 面, 面の色) の組をひとつにまとめる
    parts = [p for p in parts if p is not None and len(p[1])]
    if not parts:
        return None
    V, F, C, off = [], [], [], 0
    for v, f, c in parts:
        V.append(np.asarray(v, np.float32))
        F.append(np.asarray(f, np.int32) + off)
        C.append(np.asarray(c, np.float32))
        off += len(v)
    return np.concatenate(V), np.concatenate(F), np.concatenate(C)


def fulcrum_marker_geometry(points, weights, size=0.05, thresh=0.15):
    on = np.flatnonzero(weights > thresh)
    if len(on) == 0:
        return None
    V, F, C = [], [], []
    for k in on:
        w = float(np.clip(weights[k], 0.0, 1.0))
        V.append(OCTA_V * (size * (0.6 + 0.5 * w)) + points[k])
        F.append(OCTA_F + len(V[-1]) * (len(V) - 1))
        C.append(np.tile(MARKER_OFF * (1 - w) + MARKER_ON * w, (len(OCTA_F), 1)))
    return np.concatenate(V), np.concatenate(F), np.concatenate(C)


# ---- 出力サイズ・カメラ・床の準備 ----
out_h = int(OUT_HEIGHT) // 2 * 2
out_w = int(round(out_h * SEG_W / max(SEG_H, 1))) // 2 * 2
IMSHAPE = (out_h, out_w)

render_verts = MESH_VERTS.copy()
fulcrum_xyz = None
if SHOW_FULCRUM_MARKERS and 'FULCRUM_W' in globals() and FULCRUM_W.shape[1]:
    fulcrum_xyz = motion['joints3d'][:, FULCRUM_POINT_IDS] / 1000.0   # (T, P, 3) [m]
if VIEW_AZIMUTH_DEG:
    center = np.median(render_verts.reshape(-1, 3), axis=0)
    render_verts = rotate_about_y(render_verts, center, VIEW_AZIMUTH_DEG)
    if fulcrum_xyz is not None:
        fulcrum_xyz = rotate_about_y(fulcrum_xyz, center, VIEW_AZIMUTH_DEG)

if CAMERA_MODE == 'original' and not VIEW_AZIMUTH_DEG:
    K_RENDER = K_ORIG * np.array([[out_w / SEG_W], [out_h / SEG_H], [1.0]], np.float32)
else:
    K_RENDER = fit_camera(render_verts[::max(1, len(render_verts) // 60)], IMSHAPE)

floor = None
if SHOW_FLOOR:
    # 8c が接地から床を推定していればそれを使う（頂点の分位点より安定）
    y_floor = float(globals().get('FLOOR_Y_MM', np.nan)) / 1000.0
    if not np.isfinite(y_floor):
        y_floor = float(np.percentile(render_verts[..., 1], 99.7))
    xz = render_verts.reshape(-1, 3)[:, [0, 2]]
    span = float(max(np.ptp(np.percentile(xz[:, 0], [1, 99])),
                     np.ptp(np.percentile(xz[:, 1], [1, 99]))))
    floor = make_floor_grid([float(np.median(xz[:, 0])), float(np.median(xz[:, 1]))],
                            y_floor, half_size=max(1.2, span * 1.5 + 0.8))

body_rgb = hex2rgb(BODY_COLOR)
bg_rgb = hex2rgb(BG_COLOR)
backend = None
if RENDER_BACKEND == 'pyrender':
    try:
        backend = PyrenderBackend(IMSHAPE, bg_rgb)
        print('pyrender を使います。')
    except Exception as e:
        print('pyrender を初期化できませんでした（', repr(e), '）→ ソフトウェアレンダラを使います。')

# ---- 描画ループ ----
from PIL import Image

SILENT_MP4 = os.path.join(WORK_DIR, 'mannequin_silent.mp4')
src_reader = imageio.get_reader(SEGMENT_MP4) if SIDE_BY_SIDE else None
writer = imageio.get_writer(SILENT_MP4, fps=FPS, codec='libx264', quality=8,
                            macro_block_size=1, pixelformat='yuv420p')
for i in tqdm(range(len(render_verts)), desc='描画中'):
    extra = floor
    if fulcrum_xyz is not None:
        extra = merge_extra(floor, fulcrum_marker_geometry(fulcrum_xyz[i], FULCRUM_W[i]))
    if backend is not None:
        img = backend.render(render_verts[i], FACES, K_RENDER, IMSHAPE, body_rgb, bg_rgb, extra)
    else:
        img = render_software(render_verts[i], FACES, K_RENDER, IMSHAPE, body_rgb, bg_rgb, extra)
    if src_reader is not None:
        try:
            src = np.asarray(Image.fromarray(src_reader.get_data(i)).resize(
                (out_w, out_h), Image.BILINEAR))
            img = np.concatenate([src, img], axis=1)
        except Exception:
            pass
    writer.append_data(img)
writer.close()
if src_reader is not None:
    src_reader.close()
print('マネキン動画（無音）:', SILENT_MP4, f'({os.path.getsize(SILENT_MP4) / 1024 ** 2:.1f} MB)')

---
## 12. 音声を合成して完成 🎬

切り出しておいた音声を重ねて `mannequin_with_audio.mp4` を書き出し、その場で再生・ダウンロードします。

In [ ]:
#@title 12. 音声付き mp4 を書き出して再生 { display-mode: "form" }
import base64
from IPython.display import HTML, display

FINAL_MP4 = os.path.join(WORK_DIR, 'mannequin_with_audio.mp4')
if AUDIO_PATH:
    run_ffmpeg(['-y', '-loglevel', 'error', '-i', SILENT_MP4, '-i', AUDIO_PATH,
                '-c:v', 'copy', '-c:a', 'aac', '-shortest', FINAL_MP4])
    print('音声を合成しました。')
else:
    run_ffmpeg(['-y', '-loglevel', 'error', '-i', SILENT_MP4, '-c', 'copy', FINAL_MP4])
    print('元動画に音声が無かったため、無音で出力しました。')

size_mb = os.path.getsize(FINAL_MP4) / 1024 ** 2
print('完成:', FINAL_MP4, f'({size_mb:.1f} MB)')

if size_mb < 60:
    b64 = base64.b64encode(open(FINAL_MP4, 'rb').read()).decode()
    display(HTML(f'<video width="480" controls loop>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
else:
    print('（ファイルが大きいのでインライン再生は省略しました）')

if IN_COLAB:
    from google.colab import files
    files.download(FINAL_MP4)
    files.download(MOTION_NPZ)
print('モーションデータ:', MOTION_NPZ)

---
## 13. モーションを FBX で書き出す（Cascadeur / Blender / Unity / UE 用）

抽出したモーションを **FBX（バイナリ 7.4）** でも書き出します。追加のインストールは不要です
（FBX SDK や Blender を使わず、このセルの中で直接ファイルを書きます）。

**[Cascadeur](https://cascadeur.com/) にそのまま読み込めて、リグの調整が要らない形式**にしてあります。

| Cascadeur の要件 | この書き出し |
|---|---|
| Y 軸が上・単位は cm | Y-up / cm（`UnitScaleFactor = 1`）で出力 |
| キャラクターは +Z を向き、地面に対して直立 | カメラ座標系から変換して、正面を +Z・足元を y=0 に配置 |
| **フレーム 0 がデフォルトポーズ**（T / A ポーズ・左右対称） | `FBX_DEFAULT_POSE_FRAME` でフレーム 0 に SMPL の静止姿勢（回転ゼロ・足が地面）を入れ、モーションはフレーム 1 から |
| Quick Rigging Tool が認識する骨の名前・左右の表記が統一されていること | `FBX_BONE_NAMES='unreal'` で Unreal 準拠の名前（`pelvis` / `spine_01` / `thigh_l` / `calf_r` / `ball_l` …）にそろえ、左右は `_l` / `_r` で統一 |
| メッシュが骨にスキニングされていること | SMPL のスキニングウェイトでマネキンをバインド（`FBX_INCLUDE_MESH`） |
| ルートのスケールが 1 | 原点に固定した `root` を親に置き、スケールはすべて 1 |

中身:

| 要素 | 内容 |
|---|---|
| スケルトン | `root` ＋ SMPL の 24 関節（`pelvis` がその下のルート）。骨の長さはセル 8 で固定した共通体型から計算 |
| アニメーション | 8d が選んだ**キーフレームだけ**にキー（`FBX_KEYFRAMES_ONLY`、補間は 3 次＋自動接線）。8d を実行していなければ全フレームに線形キー。`pelvis` は位置＋回転、それ以外は回転のみ（XYZ オイラー角） |
| メッシュ | セル 10 のマネキンをデフォルトポーズで入れ、スケルトンにバインド（法線・マテリアル付き） |

* 動き（関節の回転・全身の位置）は `motion.npz` と同じで、8b / 8c の補正もすべて反映されます
* SMPL の「ポーズ補正ブレンドシェイプ」は FBX では表現できないため、肘や膝の曲げ部分の形はレンダリング動画とわずかに異なります（骨の動きは同一です）
* 指の骨はありません（SMPL は手を 1 関節で表すため）。Cascadeur の指スロットは空のままで構いません
* Blender で読み込んだときは、タイムラインの終了フレームが自動では変わりません。必要なら総フレーム数に合わせてください

In [ ]:
#@title 13. モーションを FBX で書き出す { display-mode: "form" }
FBX_INCLUDE_MESH = True  #@param {type:"boolean"}
FBX_GROUND_AT_ORIGIN = True  #@param {type:"boolean"}
FBX_DEFAULT_POSE_FRAME = True  #@param {type:"boolean"}
FBX_BONE_NAMES = "unreal"  #@param ["unreal", "smpl"]
FBX_KEYFRAMES_ONLY = True  #@param {type:"boolean"}

import struct
import zlib

# ---- FBX 7.4 バイナリの最小ライタ（Blender の io_scene_fbx/encode_bin.py と同じ書式） ----
FBX_VERSION = 7400
FBX_KTIME = 46186158000   # 1 秒あたりの FBX 時間単位
# キーの補間指定（FBX SDK の KeyAttrFlags）。全フレームにキーを打つときは線形、
# キーを間引いたときは 3 次補間 + 自動接線（Cascadeur の "adjust interpolation" 相当）。
FBX_KEY_LINEAR = 1 << 2 | 1 << 8 | 1 << 13 | 1 << 14
FBX_KEY_CUBIC = 1 << 3 | 1 << 8 | 1 << 13 | 1 << 14
_SENTINEL = b'\0' * 13
_ALWAYS_SENTINEL = {'AnimationStack', 'AnimationLayer'}


class FbxNode:
    def __init__(self, name, *props):
        self.name, self.props, self.children = name, list(props), []

    def add(self, name, *props):
        node = FbxNode(name, *props)
        self.children.append(node)
        return node


def _fbx_prop(v):
    # (型コード, バイト列) に変換する。型は値の Python 型と numpy の dtype で決める。
    if isinstance(v, tuple) and len(v) == 2 and isinstance(v[0], str):
        kind, v = v   # 型を明示したいとき: ('L', 123) / ('d', array) など
    elif isinstance(v, (bool, np.bool_)):
        kind = 'C'
    elif isinstance(v, (int, np.integer)):
        kind = 'I' if -2 ** 31 <= int(v) < 2 ** 31 else 'L'
    elif isinstance(v, (float, np.floating)):
        kind = 'D'
    elif isinstance(v, str):
        kind = 'S'
    elif isinstance(v, bytes):
        kind = 'R'
    else:
        a = np.asarray(v)
        kind = {'f': 'd', 'i': 'i', 'u': 'i', 'b': 'b'}[a.dtype.kind]
    if kind in 'SR':
        data = v.encode('utf-8') if isinstance(v, str) else v
        return kind.encode(), struct.pack('<I', len(data)) + data
    if kind in 'CYIFDL':
        fmt = {'C': '<?', 'Y': '<h', 'I': '<i', 'F': '<f', 'D': '<d', 'L': '<q'}[kind]
        return kind.encode(), struct.pack(fmt, v)
    dtype = {'d': '<f8', 'f': '<f4', 'i': '<i4', 'l': '<i8', 'b': '?'}[kind]
    a = np.ascontiguousarray(np.asarray(v).reshape(-1), dtype=dtype)
    raw = a.tobytes()
    enc = 0 if len(raw) <= 128 else 1
    if enc:
        raw = zlib.compress(raw, 1)
    return kind.encode(), struct.pack('<3I', len(a), enc, len(raw)) + raw


def _fbx_node_bytes(node, offset, is_last):
    # ノード 1 つ分のバイト列。先頭の EndOffset はファイル先頭からの絶対位置。
    props = [_fbx_prop(p) for p in node.props]
    prop_bytes = b''.join(t + d for t, d in props)
    name = node.name.encode()
    head_len = 12 + 1 + len(name) + len(prop_bytes)
    body, pos = [], offset + head_len
    for i, child in enumerate(node.children):
        b = _fbx_node_bytes(child, pos, i == len(node.children) - 1)
        body.append(b)
        pos += len(b)
    if node.children or (not node.props and not is_last) or node.name in _ALWAYS_SENTINEL:
        body.append(_SENTINEL)
        pos += len(_SENTINEL)
    head = struct.pack('<3I', pos, len(props), len(prop_bytes)) + bytes([len(name)]) + name
    return head + prop_bytes + b''.join(body)


def write_fbx_file(path, top_nodes):
    out = bytearray(b'Kaydara FBX Binary  \x00\x1a\x00' + struct.pack('<I', FBX_VERSION))
    for i, node in enumerate(top_nodes):
        out += _fbx_node_bytes(node, len(out), i == len(top_nodes) - 1)
    out += _SENTINEL
    # フッタ（FBX SDK が期待する固定値。Blender の書き出しと同じ）
    out += b'\xfa\xbc\xab\x09\xd0\xc8\xd4\x66\xb1\x76\xfb\x83\x1c\xf7\x26\x7e' + b'\0' * 4
    pad = ((len(out) + 15) & ~15) - len(out)
    out += b'\0' * (pad or 16)
    out += struct.pack('<I', FBX_VERSION) + b'\0' * 120
    out += b'\xf8\x5a\x8c\x6a\xde\xf5\xd9\x7e\xec\xe9\x0c\xe3\x75\x8f\x29\x0b'
    with open(path, 'wb') as f:
        f.write(out)
    return path


def _p70(parent, *entries):
    # Properties70 ブロック。entries = (名前, 型, ラベル, フラグ, 値...) のタプル
    p70 = parent.add('Properties70')
    for e in entries:
        p70.add('P', *e)
    return p70


def _mat_cols(m):
    # FBX の行列は列優先（平行移動が 12〜14 番目）
    return np.asarray(m, np.float64).T.reshape(-1)


def _mesh_normals(verts, faces, smooth):
    # ByPolygonVertex 用の法線 (F*3, 3)。smooth=True なら面積重みで頂点ごとに平均する。
    v, f = np.asarray(verts, np.float64), np.asarray(faces, np.int64)
    a, b, c = v[f[:, 0]], v[f[:, 1]], v[f[:, 2]]
    fn = np.cross(b - a, c - a)   # 長さ = 面積の 2 倍 = 面積重み
    if not smooth:
        n = np.repeat(fn, 3, axis=0)
    else:
        vn = np.zeros_like(v)
        for k in range(3):
            np.add.at(vn, f[:, k], fn)
        n = vn[f.reshape(-1)]
    return n / np.maximum(np.linalg.norm(n, axis=-1, keepdims=True), 1e-12)


def build_motion_fbx(path, names, parents, rest_joints_cm, local_euler_deg, root_pos_cm, fps,
                     mesh=None, body_color=(0.85, 0.82, 0.78), take_name='Take 001',
                     root_bone_name='root', smooth_normals=True, key_frames=None):
    # names/parents: 骨の名前と親 (J,) / rest_joints_cm: T ポーズの関節位置 (J, 3)
    # local_euler_deg: 各骨の親に対する回転 (T, J, 3)、XYZ オイラー角 [度]（行列 = Rz·Ry·Rx）
    # root_pos_cm: ルート骨の位置 (T, 3)
    # mesh: None または (頂点 (V,3) [cm, T ポーズ], 面 (F,3), スキンウェイト (V,J))
    # root_bone_name: 原点に固定する親骨の名前（None なら作らない）
    # key_frames: キーを打つフレーム番号（None なら全フレーム）。間引いた場合は 3 次補間で書く。
    J, T = len(names), len(local_euler_deg)
    uid = iter(range(1_000_000, 10_000_000))
    ids = lambda: ('L', next(uid))
    objects, conns = FbxNode('Objects'), FbxNode('Connections')
    counts = {}

    def obj(kind, *props):
        counts[kind] = counts.get(kind, 0) + 1
        return objects.add(kind, *props)

    def connect(child, parent, prop=None):
        if prop is None:
            conns.add('C', 'OO', child, parent)
        else:
            conns.add('C', 'OP', child, parent, prop)

    def model(mid, name, kind, t, r=(0.0, 0.0, 0.0)):
        m = obj('Model', mid, name + '\x00\x01Model', kind)
        m.add('Version', 232)
        _p70(m,
             ('RotationActive', 'bool', '', '', 1),
             ('InheritType', 'enum', '', '', 1),
             ('DefaultAttributeIndex', 'int', 'Integer', '', 0),
             ('Lcl Translation', 'Lcl Translation', '', 'A', *map(float, t)),
             ('Lcl Rotation', 'Lcl Rotation', '', 'A', *map(float, r)),
             ('Lcl Scaling', 'Lcl Scaling', '', 'A', 1.0, 1.0, 1.0))
        m.add('Shading', True)
        m.add('Culling', 'CullingOff')
        return m

    def limb(name, t, r=(0.0, 0.0, 0.0)):
        attr_id, bid = ids(), ids()
        a = obj('NodeAttribute', attr_id, name + '\x00\x01NodeAttribute', 'LimbNode')
        _p70(a, ('Size', 'double', 'Number', '', 3.0))
        a.add('TypeFlags', 'Skeleton')
        model(bid, name, 'LimbNode', t, r)
        connect(attr_id, bid)
        return bid

    # ---- 骨（LimbNode） ----
    # ゲームエンジン（Unreal など）と同じく、原点に固定の 'root' を置いてその下に骨を並べる。
    root_id = limb(root_bone_name, (0.0, 0.0, 0.0)) if root_bone_name else ('L', 0)
    if root_bone_name:
        connect(root_id, ('L', 0))
    bone_ids = []
    for j in range(J):
        p = parents[j]
        if p < 0:
            t, r = root_pos_cm[0], local_euler_deg[0, j]
        else:
            t, r = rest_joints_cm[j] - rest_joints_cm[p], local_euler_deg[0, j]
        bid = limb(names[j], t, r)
        connect(bid, bone_ids[p] if p >= 0 else root_id)
        bone_ids.append(bid)

    # ---- スキン付きメッシュ（任意） ----
    if mesh is not None:
        verts, faces, weights = mesh
        mesh_id, geo_id, mat_id, skin_id, pose_id = ids(), ids(), ids(), ids(), ids()
        model(mesh_id, 'Mannequin', 'Mesh', (0.0, 0.0, 0.0))
        connect(mesh_id, ('L', 0))

        g = obj('Geometry', geo_id, 'Mannequin\x00\x01Geometry', 'Mesh')
        g.add('Vertices', ('d', np.asarray(verts, np.float64)))
        pvi = np.asarray(faces, np.int32).copy()
        pvi[:, -1] = ~pvi[:, -1]   # 多角形の最後の頂点はビット反転で区切りを表す
        g.add('PolygonVertexIndex', ('i', pvi))
        g.add('GeometryVersion', 124)
        ln = g.add('LayerElementNormal', 0)
        ln.add('Version', 102)
        ln.add('Name', '')
        ln.add('MappingInformationType', 'ByPolygonVertex')
        ln.add('ReferenceInformationType', 'Direct')
        ln.add('Normals', ('d', _mesh_normals(verts, faces, smooth_normals)))
        lm = g.add('LayerElementMaterial', 0)
        lm.add('Version', 101)
        lm.add('Name', '')
        lm.add('MappingInformationType', 'AllSame')
        lm.add('ReferenceInformationType', 'IndexToDirect')
        lm.add('Materials', ('i', np.zeros(1, np.int32)))
        layer = g.add('Layer', 0)
        layer.add('Version', 100)
        for kind in ('LayerElementNormal', 'LayerElementMaterial'):
            le = layer.add('LayerElement')
            le.add('Type', kind)
            le.add('TypedIndex', 0)
        connect(geo_id, mesh_id)

        mat = obj('Material', mat_id, 'MannequinMat\x00\x01Material', '')
        mat.add('Version', 102)
        mat.add('ShadingModel', 'Phong')
        mat.add('MultiLayer', 0)
        c = tuple(float(x) for x in body_color)
        _p70(mat,
             ('DiffuseColor', 'Color', '', 'A', *c),
             ('DiffuseFactor', 'Number', '', 'A', 1.0),
             ('SpecularFactor', 'Number', '', 'A', 0.1),
             ('Shininess', 'Number', '', 'A', 10.0))
        connect(mat_id, mesh_id)

        skin = obj('Deformer', skin_id, 'Skin\x00\x01Deformer', 'Skin')
        skin.add('Version', 101)
        skin.add('Link_DeformAcuracy', 50.0)
        connect(skin_id, geo_id)

        bind_world = []
        for j in range(J):
            m = np.eye(4)
            m[:3, 3] = rest_joints_cm[j]   # T ポーズでは全骨の回転が単位行列
            bind_world.append(m)
            idx = np.flatnonzero(weights[:, j] > 1e-4)
            cid = ids()
            cl = obj('Deformer', cid, names[j] + '\x00\x01SubDeformer', 'Cluster')
            cl.add('Version', 100)
            cl.add('UserData', '', '')
            if len(idx):
                cl.add('Indexes', ('i', idx.astype(np.int32)))
                cl.add('Weights', ('d', weights[idx, j].astype(np.float64)))
            # Transform は「骨の空間から見たメッシュ」（Blender の書き出しと同じ解釈）
            cl.add('Transform', ('d', _mat_cols(np.linalg.inv(m))))
            cl.add('TransformLink', ('d', _mat_cols(m)))
            connect(cid, skin_id)
            connect(bone_ids[j], cid)

        pose = obj('Pose', pose_id, 'BindPose\x00\x01Pose', 'BindPose')
        pose.add('Type', 'BindPose')
        pose.add('Version', 100)
        pose.add('NbPoseNodes', J + 1)
        for nid, m in [(mesh_id, np.eye(4))] + list(zip(bone_ids, bind_world)):
            pn = pose.add('PoseNode')
            pn.add('Node', nid)
            pn.add('Matrix', ('d', _mat_cols(m)))

    # ---- アニメーション ----
    ktimes_all = np.round(np.arange(T) * (FBX_KTIME / float(fps))).astype(np.int64)
    if key_frames is None:
        key_idx, key_flags = np.arange(T), FBX_KEY_LINEAR
    else:
        key_idx = np.unique(np.clip(np.asarray(key_frames, np.int64), 0, T - 1))
        key_idx = np.unique(np.concatenate([[0], key_idx, [T - 1]]))
        key_flags = FBX_KEY_CUBIC if len(key_idx) < T else FBX_KEY_LINEAR
    ktimes = ktimes_all[key_idx]
    n_keys = len(key_idx)
    t_end = int(ktimes_all[-1])
    stack_id, layer_id = ids(), ids()
    st = obj('AnimationStack', stack_id, take_name + '\x00\x01AnimStack', '')
    _p70(st, ('LocalStop', 'KTime', 'Time', '', ('L', t_end)),
         ('ReferenceStop', 'KTime', 'Time', '', ('L', t_end)))
    obj('AnimationLayer', layer_id, 'BaseLayer\x00\x01AnimLayer', '')
    connect(layer_id, stack_id)

    def anim_channel(target, prop, values):
        values = np.asarray(values, np.float64)[key_idx]   # キーフレームだけを書き出す
        cn_id = ids()
        cn = obj('AnimationCurveNode', cn_id, prop[4].upper() + '\x00\x01AnimCurveNode', '')
        _p70(cn, *[(f'd|{ax}', 'Number', '', 'A', float(values[0, k]))
                   for k, ax in enumerate('XYZ')])
        connect(cn_id, layer_id)
        connect(cn_id, target, prop)
        for k, ax in enumerate('XYZ'):
            cv_id = ids()
            cv = obj('AnimationCurve', cv_id, '\x00\x01AnimCurve', '')
            cv.add('Default', float(values[0, k]))
            cv.add('KeyVer', 4008)
            cv.add('KeyTime', ('l', ktimes))
            cv.add('KeyValueFloat', ('f', values[:, k].astype(np.float32)))
            cv.add('KeyAttrFlags', ('i', np.array([key_flags], np.int32)))
            cv.add('KeyAttrDataFloat', ('f', np.array([0, 0, 9.419963346924634e-30, 0], np.float32)))
            cv.add('KeyAttrRefCount', ('i', np.array([n_keys], np.int32)))
            connect(cv_id, cn_id, f'd|{ax}')

    for j in range(J):
        if parents[j] < 0:
            anim_channel(bone_ids[j], 'Lcl Translation', root_pos_cm)
        anim_channel(bone_ids[j], 'Lcl Rotation', local_euler_deg[:, j])

    # ---- ヘッダ・グローバル設定・定義 ----
    header = FbxNode('FBXHeaderExtension')
    header.add('FBXHeaderVersion', 1003)
    header.add('FBXVersion', FBX_VERSION)
    header.add('EncryptionType', 0)
    ts = header.add('CreationTimeStamp')
    for k, v in [('Version', 1000), ('Year', 1970), ('Month', 1), ('Day', 1), ('Hour', 10),
                 ('Minute', 0), ('Second', 0), ('Millisecond', 0)]:
        ts.add(k, v)
    header.add('Creator', 'nlf mp4_to_mannequin')

    fps_modes = {120.0: 1, 100.0: 2, 60.0: 3, 50.0: 4, 48.0: 5, 30.0: 6, 25.0: 10, 24.0: 11,
                 96.0: 15, 72.0: 16}
    time_mode = next((m for f, m in fps_modes.items() if abs(f - fps) < 1e-3), 14)
    gs = FbxNode('GlobalSettings')
    gs.add('Version', 1000)
    _p70(gs,
         ('UpAxis', 'int', 'Integer', '', 1), ('UpAxisSign', 'int', 'Integer', '', 1),
         ('FrontAxis', 'int', 'Integer', '', 2), ('FrontAxisSign', 'int', 'Integer', '', 1),
         ('CoordAxis', 'int', 'Integer', '', 0), ('CoordAxisSign', 'int', 'Integer', '', 1),
         ('OriginalUpAxis', 'int', 'Integer', '', 1), ('OriginalUpAxisSign', 'int', 'Integer', '', 1),
         ('UnitScaleFactor', 'double', 'Number', '', 1.0),        # 1 単位 = 1 cm
         ('OriginalUnitScaleFactor', 'double', 'Number', '', 1.0),
         ('TimeMode', 'enum', '', '', time_mode),
         ('TimeSpanStart', 'KTime', 'Time', '', ('L', 0)),
         ('TimeSpanStop', 'KTime', 'Time', '', ('L', t_end)),
         ('CustomFrameRate', 'double', 'Number', '', float(fps)))

    docs = FbxNode('Documents')
    docs.add('Count', 1)
    doc = docs.add('Document', ids(), 'Scene', 'Scene')
    _p70(doc, ('SourceObject', 'object', '', ''),
         ('ActiveAnimStackName', 'KString', '', '', take_name))
    doc.add('RootNode', ('L', 0))

    defs = FbxNode('Definitions')
    defs.add('Version', 100)
    defs.add('Count', sum(counts.values()) + 1)
    defs.add('ObjectType', 'GlobalSettings').add('Count', 1)
    for kind, n in counts.items():
        defs.add('ObjectType', kind).add('Count', n)

    top = [header,
           FbxNode('FileId', b'\x28\xb3\x2a\xeb\xb6\x24\xcc\xc2\xbf\xc8\xb0\x2a\xa9\x2b\xfc\xf1'),
           FbxNode('CreationTime', '1970-01-01 10:00:00:000'),
           FbxNode('Creator', 'nlf mp4_to_mannequin'),
           gs, docs, FbxNode('References'), defs, objects, conns,
           FbxNode('Takes')]
    takes = top[-1]
    takes.add('Current', take_name)
    tk = takes.add('Take', take_name)
    tk.add('FileName', take_name.replace(' ', '_') + '.tak')
    tk.add('LocalTime', ('L', 0), ('L', t_end))
    tk.add('ReferenceTime', ('L', 0), ('L', t_end))
    return write_fbx_file(path, top)


# ---- SMPL のモーション -> FBX の骨アニメーション ----
# カメラ座標系（x=右 / y=下 / z=奥）を、FBX の標準である Y-up（x=右 / y=上 / z=手前）に直す。
# x 軸まわりの 180 度回転なので、全身の向き（ルートの回転）と位置にだけ掛ければよい。
CAM_TO_YUP = np.diag([1.0, -1.0, -1.0])


def smpl_forward_kinematics(pose, trans, rest_joints, parents):
    # SMPL と同じ順運動学: 関節 j の位置 = 親の位置 + 親の大域回転 × (静止姿勢での親からの差分)
    T, J = pose.shape[:2]
    R = Rotation.from_rotvec(pose.reshape(-1, 3)).as_matrix().reshape(T, J, 3, 3)
    G, P = np.empty_like(R), np.empty((T, J, 3))
    for j, p in enumerate(parents):
        if p < 0:
            G[:, j], P[:, j] = R[:, j], rest_joints[j] + trans
        else:
            G[:, j] = G[:, p] @ R[:, j]
            P[:, j] = P[:, p] + G[:, p] @ (rest_joints[j] - rest_joints[p])
    return P


def smpl_to_fbx_channels(pose, trans, rest_joints, shift):
    # 戻り値: 各骨の XYZ オイラー角 [度] (T, J, 3) と、ルート骨の位置 [cm] (T, 3)
    T, J = pose.shape[:2]
    R = Rotation.from_rotvec(pose.reshape(-1, 3)).as_matrix().reshape(T, J, 3, 3)
    R[:, 0] = CAM_TO_YUP @ R[:, 0]
    # 小文字 'xyz'（固定軸）= 行列 Rz·Ry·Rx で、FBX の回転順序 XYZ と同じ
    euler = Rotation.from_matrix(R.reshape(-1, 3, 3)).as_euler('xyz').reshape(T, J, 3)
    euler = np.rad2deg(np.unwrap(euler, axis=0))   # ±180 度の折り返しで補間が 1 回転しないように
    root = (rest_joints[0] + trans) @ CAM_TO_YUP.T + shift
    return euler, root * 100.0


# SMPL の 24 関節を Unreal / Cascadeur で標準的な骨名に対応させる（左右は _l / _r で統一）。
# Cascadeur の Quick Rigging Tool はこの命名の骨格を自動認識します。
UE_BONE_NAMES = [
    'pelvis', 'thigh_l', 'thigh_r', 'spine_01', 'calf_l', 'calf_r', 'spine_02',
    'foot_l', 'foot_r', 'spine_03', 'ball_l', 'ball_r', 'neck_01', 'clavicle_l',
    'clavicle_r', 'head', 'upperarm_l', 'upperarm_r', 'lowerarm_l', 'lowerarm_r',
    'hand_l', 'hand_r', 'middle_01_l', 'middle_01_r']

# ---- 静止姿勢（デフォルトポーズ）の関節・頂点を体モデルから作る ----
fbx_repose_fn, _ = get_repose_fn()
assert fbx_repose_fn is not None, '体モデルが使えないため FBX を書き出せません。'
betas_fbx = motion['betas'][0].astype(np.float32)   # セル 8 / 8b で全フレーム共通にした体型
v_rest, j_rest = repose(np.zeros((1, N_JOINTS, 3), np.float32), betas_fbx[None],
                        np.zeros((1, 3), np.float32), fbx_repose_fn)
v_rest, j_rest = v_rest[0] / 1000.0, j_rest[0] / 1000.0   # [m]

pose_fbx = motion['pose'].astype(np.float64)
trans_fbx = motion['trans'].astype(np.float64)
fk_err = np.abs(smpl_forward_kinematics(pose_fbx, trans_fbx, j_rest, SMPL_PARENTS) * 1000.0
                - motion['joints3d']).max()
print(f'骨アニメーションの自己検証: motion.npz の関節との最大差 {fk_err:.2f} mm')
if fk_err > 20.0:
    print('⚠️ 差が大きめです（8b のリフィット後の平滑化や、再ポーズが使えなかった場合に起こります）。')

# ---- 置き場所: 床を y=0 に、水平方向は全身の平均位置を原点に ----
shift = np.zeros(3)
if FBX_GROUND_AT_ORIGIN:
    floor_mm = float(globals().get('FLOOR_Y_MM', np.nan))
    if not np.isfinite(floor_mm):
        floor_mm = float(np.percentile(motion['vertices3d'][..., 1], 99.7))
    shift[1] = floor_mm / 1000.0   # y を反転するので、カメラ座標の床の高さ F は -F になる
    root_yup = (j_rest[0] + trans_fbx) @ CAM_TO_YUP.T
    shift[[0, 2]] = -root_yup[:, [0, 2]].mean(0)
euler_fbx, root_cm = smpl_to_fbx_channels(pose_fbx, trans_fbx, j_rest, shift)

# ---- マネキンのメッシュ（セル 10 と同じ形）とスキンウェイト ----
fbx_mesh = None
if FBX_INCLUDE_MESH and 'FACES' in globals():
    lbs = try_get_lbs_weights(nlf_model, BODY_MODEL_NAME, len(v_rest))
    if lbs is None and BODY_MODEL is not None:
        lbs = BODY_MODEL.weights.detach().float().cpu().numpy()
    if lbs is None:
        # スキンウェイトが取れなければ、各頂点を最も近い関節に 100% 割り当てる
        near = np.linalg.norm(v_rest[:, None] - j_rest[None], axis=-1).argmin(1)
        lbs = np.eye(N_JOINTS, dtype=np.float32)[near]
    w = lbs[VERTEX_MAP]
    w = w / np.maximum(w.sum(1, keepdims=True), 1e-8)
    fbx_mesh = (v_rest[VERTEX_MAP] * 100.0, FACES, w)
elif FBX_INCLUDE_MESH:
    print('⚠️ セル 10 のマネキンがまだ無いので、骨だけの FBX にします。')

# ---- フレーム 0 にデフォルトポーズ（回転ゼロ・足を地面に）を 1 枚入れる ----
# Cascadeur など「フレーム 0 が T/A ポーズであること」を前提にするツール向け。
# 姿勢は SMPL の静止姿勢そのもの（左右対称・直立・正面が +Z）なので、
# 読み込んだ側でリグを組み直さなくてもそのまま使えます。
# ---- 8d で選んだキーフレームだけを書き出す（Cascadeur の "using only key frames"）----
fbx_keys = None
if FBX_KEYFRAMES_ONLY and len(np.atleast_1d(globals().get('KEY_FRAMES', []))) > 0:
    fbx_keys = np.asarray(KEY_FRAMES, np.int64)

if FBX_DEFAULT_POSE_FRAME:
    ground = (fbx_mesh[0][:, 1].min() / 100.0 if fbx_mesh is not None else j_rest[:, 1].min())
    rest_root = (j_rest[0] - [0.0, ground, 0.0]) * 100.0
    euler_fbx = np.concatenate([np.zeros((1, N_JOINTS, 3)), euler_fbx])
    root_cm = np.concatenate([rest_root[None], root_cm])
    if fbx_keys is not None:
        fbx_keys = np.concatenate([[0], fbx_keys + 1])   # フレーム 0 のデフォルトポーズぶんずらす

MOTION_FBX = os.path.join(WORK_DIR, 'motion.fbx')
build_motion_fbx(MOTION_FBX,
                 UE_BONE_NAMES if FBX_BONE_NAMES == 'unreal' else SMPL_JOINT_NAMES,
                 SMPL_PARENTS, j_rest * 100.0, euler_fbx, root_cm, FPS, mesh=fbx_mesh,
                 body_color=hex2rgb(BODY_COLOR) if 'BODY_COLOR' in globals() else (0.85, 0.82, 0.78),
                 smooth_normals=bool(globals().get('use_smpl_mesh', False)),
                 key_frames=fbx_keys)
print('FBX を書き出しました:', MOTION_FBX, f'({os.path.getsize(MOTION_FBX) / 1024 ** 2:.1f} MB)')
print(f'  骨 {N_JOINTS} 本（root + {FBX_BONE_NAMES} 命名） / {len(euler_fbx)} フレーム @ {FPS:g} fps / '
      + (f'スキン付きメッシュ（頂点 {len(fbx_mesh[0])} / 面 {len(fbx_mesh[1])}）'
         if fbx_mesh is not None else 'メッシュなし'))
if fbx_keys is not None and len(fbx_keys) < len(euler_fbx):
    print(f'  キーは {len(fbx_keys)} 本だけ（3 次補間）: Cascadeur でそのまま編集できます。')
if FBX_DEFAULT_POSE_FRAME:
    print('  フレーム 0 = デフォルトポーズ、フレーム 1 以降がモーションです。')

if IN_COLAB:
    from google.colab import files
    files.download(MOTION_FBX)

---
## 14. うまくいかないときは

| 症状 | 対処 |
|---|---|
| `CUDA out of memory` | `BATCH_SIZE` を 1〜2 に、`MAX_HEIGHT` を 480 に下げる |
| GPU が無いというエラー | Colab のランタイムを GPU に変更して最初から実行し直す |
| 人物が検出されない | 区間を人物が大きく写っているところに変える。セル 7 の `DETECTOR_THRESHOLD` を 0.15 程度に下げる |
| 別の人に乗り移る | `PERSON_SELECT` を変える。セル 7 の追跡しきい値 `1.5`（m）を小さくする |
| 体が細かく震える | `NUM_AUG` を 5 に上げる（推定ノイズを発生源で減らす）。`ROOT_CUTOFF_HZ` を 2.0 に下げる。8d の `KEY_ANGLE_TOL_DEG` を 3 程度に上げるとキーが減って振動も減ります |
| マネキンが大きくなったり小さくなったりする | `DEPTH_SCALE_FIX=True` を確認（セル 8 の「奥行き補正」の表示で揺れが減っているか見る）。まだブレるなら `DEPTH_EXTRA_SMOOTH=True` |
| 動きがぼやける・キレがなくなる | `SMOOTH_CUTOFF_HZ` を 8〜10 に上げる。`ROOT_CUTOFF_HZ` も上げる |
| 足が地面を滑る | `FOOT_LOCK=True` のまま `ROOT_MOTION='locked'` にする。8c のグラフで接地区間（灰色）が出ていなければ `CONTACT_HEIGHT_MM` / `CONTACT_SPEED_MMPS` を大きくする。さらに 8d の `FULCRUM_FIX_HZ` を 4〜5 に上げる |
| 8d のあと、ひざ・ひじがカクつく | `FULCRUM_FIX_HZ` を 1.5 程度に下げる（IK に細かい揺れまで直させない）。それでも気になるなら `AUTOPOSE_LOCK='feet'` |
| 手が床に着く動きなのに支点にならない | `AUTOPOSE_LOCK='feet_and_hands'` にして、8c の `CONTACT_HEIGHT_MM` を大きくする |
| 動きがキビキビしすぎ／逆に丸まりすぎ | 8d の `KEY_ANGLE_TOL_DEG` を下げる（キーが増えて原型に近づく）／上げる（キーが減って滑らかになる） |
| キーフレームを間引きたくない | 8d の `USE_KEYFRAMES_ONLY=False`（FBX は全フレームにキーを打ちます） |
| 足が地面にめり込む / 浮く | `CONTACT_HEIGHT_MM` を調整。接地が全く検出されていないと垂直補正も効きません |
| その場で踊ってほしくない（移動を残したい） | `ROOT_MOTION='full'` |
| GPU が余っている | `BATCH_SIZE` → `NUM_AUG` の順に上げる（クロップ枚数 = フレーム数 × 人数 × NUM_AUG） |
| マネキンが小さい / 見切れる | `CAMERA_MODE='fit'` にする。`fit_camera` の `margin` を調整する |
| 全身が入っていない動画 | NLF は部分的な人体でも推定しますが、全身が写っている区間のほうが安定します |
| FBX を読み込むとマネキンが寝ている / 小さい | 読み込み側の軸設定を「Y-up」、単位を「cm」にする（Blender・Cascadeur は既定のままで OK） |
| Cascadeur の Quick Rigging Tool が骨格を認識しない | `FBX_BONE_NAMES='unreal'`（既定）で書き出す。ミラー設定は `_l` / `_r` を指定する |
| 処理が遅い | `TARGET_FPS` を 15 に、`MAX_HEIGHT` を 480 に下げる |

### 出力ファイル

| ファイル | 内容 |
|---|---|
| `nlf_mannequin/motion.npz` | 抽出したモーションデータ（`pose` (T,24,3) 回転ベクトル、`betas`、`trans`、`joints3d`、`vertices3d`、`ground_offset_mm`、`key_frames`、`fulcrum_weights`、`fps` など）。セル 8 / 8b / 8c / 8d のどれを実行しても最新の状態で保存し直されます |
| `nlf_mannequin/mannequin_silent.mp4` | マネキン動画（無音） |
| `nlf_mannequin/mannequin_with_audio.mp4` | **最終出力**（音声付き） |
| `nlf_mannequin/motion.fbx` | モーションの FBX（`root` ＋ SMPL 24 関節のスケルトン＋スキン付きマネキン＋アニメーション。Y-up / cm。Cascadeur にそのまま読み込めます） |

### 参考

* NLF: <https://github.com/isarandi/nlf> — [NeurIPS 2024 論文](https://arxiv.org/abs/2407.07532)
* NLF v0.3.2 リリース: <https://github.com/isarandi/nlf/releases/tag/v0.3.2>
* SMPLFitter: <https://github.com/isarandi/smplfitter>

```
@article{sarandi2024nlf,
    title   = {Neural Localizer Fields for Continuous 3D Human Pose and Shape Estimation},
    author  = {S\'ar\'andi, Istv\'an and Pons-Moll, Gerard},
    journal = {Advances in Neural Information Processing Systems (NeurIPS)},
    year    = {2024}
}
```